## **Philadelphia Restaurant Customer Experience Intelligence & Demand Forecasting**
George Danut ASU ID: [redacted]



## Exectuive Summary

#### This project proposes the development of a Restaurant Customer Experience Intelligence and Demand Forecasting framework using Yelp business, review, user, tip, and check-in data. The project will focus primarily on restaurants in Philadelphia, Pennsylvania, one of the largest restaurant markets represented in the supplied dataset.

#### The objective is to transform Yelp's unstructured customer feedback into actionable business intelligence that restaurant operators can use to understand the drivers of customer satisfaction, identify emerging operational problems, benchmark competitors, and anticipate future customer activity. Rather than relying solely on average star ratings, natural language processing (NLP) will be used to identify what customers discuss and how they feel about specific aspects of their restaurant experiences, including food quality, service, wait times, pricing, ambience, cleanliness, and convenience.

## Data Sources

#### The project will use five interconnected JSON files from the Yelp Academic Dataset:

*   Business Data: Restaurant characteristics including location, categories, star ratings, review counts, operating status, hours, and business attributes.
*   Review Data: Customer review text, star ratings, review dates, and engagement measures. This will serve as the project's primary unstructured-data source.
*  User Data: Reviewer characteristics such as review activity, Yelp tenure, average ratings, elite status, fans, and compliment measures.
* Tip Data: Short-form customer comments and recommendations that provide an additional source of unstructured feedback.
* Check-In Data: Timestamped customer check-in activity that will be aggregated over time and used as a proxy for customer activity and demand.

The datasets can be integrated primarily through business_id and user_id, allowing customer language, restaurant characteristics, reviewer characteristics, and temporal activity to be analyzed within a unified framework.

## Load Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# File paths for each dataset

import os


In [ ]:
base_path = 'data/yelp_dataset_new'

business_file_path = os.path.join(
    base_path,
    'yelp_academic_dataset_business.json'
)

review_file_path = os.path.join(
    base_path,
    'yelp_academic_dataset_review.json'
)

user_file_path = os.path.join(
    base_path,
    'yelp_academic_dataset_user.json'
)

checkin_file_path = os.path.join(
    base_path,
    'yelp_academic_dataset_checkin.json'
)

tip_file_path = os.path.join(
    base_path,
    'yelp_academic_dataset_tip.json'
)

In [ ]:
## Import libraries

import pandas as pd
import numpy as np
import json
from pathlib import Path

In [ ]:
## Check Files

file_paths = {
    'Business': business_file_path,
    'Review': review_file_path,
    'User': user_file_path,
    'Check-in': checkin_file_path,
    'Tip': tip_file_path
}

for dataset_name, file_path in file_paths.items():

    if os.path.exists(file_path):
        print(f'{dataset_name:<10}: FOUND')

    else:
        print(f'{dataset_name:<10}: NOT FOUND')

In [ ]:
import os

search_root = 'data/'
target_folder = 'yelp_dataset_new'

for root, dirs, files in os.walk(search_root):

    if target_folder in dirs:
        folder_path = os.path.join(
            root,
            target_folder
        )

        print("Found folder:")
        print(folder_path)

In [ ]:
with open(business_file_path, 'r', encoding='utf-8') as f:
    for i in range(5):
        print(repr(f.readline()))

In [ ]:
import os

print("File size in bytes:", os.path.getsize(business_file_path))
print("File size in MB:", os.path.getsize(business_file_path) / (1024**2))

In [ ]:
## Load Business Data
business = pd.read_json(
    business_file_path
)

In [ ]:
business.head()

In [ ]:
business.shape

In [ ]:
business.info()

In [ ]:
business.columns.tolist()

In [ ]:
## Inspect most common cities

business['city'].value_counts().head(20)

In [ ]:
## Check Category field

business['categories'].head(20)

In [ ]:
## Create an exact restaurant flag

def is_restaurant(categories):
    """
    Return True if 'Restaurants' is one of the Yelp categories.
    """

    if pd.isna(categories):
        return False

    category_list = [
        category.strip()
        for category in categories.split(',')
    ]

    return 'Restaurants' in category_list

In [ ]:
business['is_restaurant'] = (
    business['categories']
    .apply(is_restaurant)
)

In [ ]:
## Check
business['is_restaurant'].value_counts()

In [ ]:
## Filter to Philadelphia

philly_business = business[
    (business['city'].str.strip().str.lower() == 'philadelphia') &
    (business['state'].str.strip().str.upper() == 'PA') &
    (business['is_restaurant'])
].copy()

In [ ]:
print(
    f'Philadelphia restaurants: '
    f'{len(philly_business):,}'
)

In [ ]:
philly_business[
    [
        'business_id',
        'name',
        'city',
        'state',
        'categories',
        'stars',
        'review_count',
        'is_open'
    ]
].head(10)

In [ ]:
## Validate Filter

print(philly_business['city'].value_counts())
print(philly_business['state'].value_counts())
print(philly_business['is_restaurant'].value_counts())

In [ ]:
## Extract Business IDs

philly_business_ids = set(
    philly_business['business_id']
)

len(philly_business_ids)

In [ ]:
## rename fields

philly_business = philly_business.rename(
    columns={
        'stars': 'business_stars',
        'review_count': 'business_review_count'
    }
)

philly_business.columns.tolist()

In [ ]:
## Profile Philadelphia Restaurants

philly_business[
    [
        'business_stars',
        'business_review_count'
    ]
].describe()

In [ ]:
# Check Operating Status
philly_business['is_open'].value_counts()

In [ ]:
## Load Review Data
reviews = pd.read_json(
    review_file_path
)

In [ ]:
## Check Dimensions and Structure

print(f'Total reviews: {reviews.shape[0]:,}')
print(f'Total columns: {reviews.shape[1]}')

In [ ]:
reviews.head()

In [ ]:
reviews.info()

In [ ]:
reviews.columns.tolist()

In [ ]:
## Filter to Philadelphia Restaurant reviews

philly_reviews = reviews[
    reviews['business_id'].isin(philly_business_ids)
].copy()

In [ ]:
print(f'All Yelp reviews: {len(reviews):,}')
print(f'Philadelphia restaurant reviews: {len(philly_reviews):,}')

In [ ]:
## Check if our restaurants have reviews

restaurants_with_reviews = (
    philly_reviews['business_id']
    .nunique()
)

print(f'Philadelphia restaurants: {len(philly_business):,}')
print(f'Restaurants with review data: {restaurants_with_reviews:,}')

In [ ]:
## Rename Fields
philly_reviews = philly_reviews.rename(
    columns={
        'stars': 'review_stars'
    }
)

In [ ]:
# Inspect stats of reviews of sample size

philly_reviews[
    'review_stars'
].describe()

In [ ]:
philly_reviews[
    'review_stars'
].value_counts().sort_index()

In [ ]:
(
    philly_reviews[
        'review_stars'
    ]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

In [ ]:
## Reviews Per Restaurant

reviews_per_restaurant = (
    philly_reviews
    .groupby('business_id')
    .size()
    .reset_index(
        name='observed_reviews'
    )
)

In [ ]:
reviews_per_restaurant[
    'observed_reviews'
].describe()

In [ ]:
reviews_per_restaurant.sort_values(
    'observed_reviews'
).head(20)

In [ ]:
reviews_per_restaurant.sort_values(
    'observed_reviews',
    ascending=False
).head(20)

In [ ]:
## Test a minimum Review Threashold

review_threshold_summary = pd.DataFrame({

    'Minimum_Reviews': [
        1,
        5,
        10,
        20,
        30,
        50,
        100
    ]
})

In [ ]:
review_threshold_summary[
    'Restaurant_Count'
] = review_threshold_summary[
    'Minimum_Reviews'
].apply(
    lambda threshold:
    (
        reviews_per_restaurant[
            'observed_reviews'
        ] >= threshold
    ).sum()
)

In [ ]:
review_threshold_summary[
    'Percent_of_Restaurants'
] = (
    review_threshold_summary[
        'Restaurant_Count'
    ]
    / len(reviews_per_restaurant)
    * 100
).round(2)

In [ ]:
review_threshold_summary

In [ ]:
## Check Missing Text
missing_review_text = (
    philly_reviews[
        'text'
    ].isna().sum()
)

print(
    f'Missing review text: '
    f'{missing_review_text:,}'
)

In [ ]:
## Check for blank text

empty_review_text = (
    philly_reviews[
        'text'
    ]
    .fillna('')
    .str.strip()
    .eq('')
    .sum()
)

print(
    f'Empty review text: '
    f'{empty_review_text:,}'
)

In [ ]:
## Check some Actual Reviews
for review in philly_reviews['text'].head(5):
    print(review)
    print('-' * 80)

In [ ]:
## Character Count Feature

philly_reviews[
    'review_character_count'
] = (
    philly_reviews[
        'text'
    ]
    .fillna('')
    .str.len()
)

In [ ]:
## Word Count Feature

philly_reviews[
    'review_word_count'
] = (
    philly_reviews[
        'text'
    ]
    .fillna('')
    .str.split()
    .str.len()
)

In [ ]:
philly_reviews[
    [
        'review_character_count',
        'review_word_count'
    ]
].describe()

In [ ]:
## Convert Date

philly_reviews[
    'date'
] = pd.to_datetime(
    philly_reviews[
        'date'
    ],
    errors='coerce'
)

In [ ]:
print(
    'Earliest review:',
    philly_reviews['date'].min()
)

print(
    'Latest review:',
    philly_reviews['date'].max()
)

In [ ]:
print(
    'Missing/invalid dates:',
    philly_reviews['date'].isna().sum()
)

In [ ]:
## Create Temporal Variables
philly_reviews[
    'year'
] = philly_reviews[
    'date'
].dt.year

philly_reviews[
    'month'
] = philly_reviews[
    'date'
].dt.month

philly_reviews[
    'year_month'
] = philly_reviews[
    'date'
].dt.to_period('M')

philly_reviews[
    'day_of_week'
] = philly_reviews[
    'date'
].dt.day_name()

In [ ]:
# Check - will use this for any potential time-seires analysis
philly_reviews[
    [
        'date',
        'year',
        'month',
        'year_month',
        'day_of_week'
    ]
].head()

In [ ]:
## Create Satisfaction Classification
negative_stars = [1, 2]
neutral_stars = [3]
positive_stars = [4, 5]

In [ ]:
def classify_star_rating(stars):
    """
    Convert a Yelp review star rating into a
    customer satisfaction category.
    """

    if stars in negative_stars:
        return 'Negative'

    elif stars in neutral_stars:
        return 'Neutral'

    elif stars in positive_stars:
        return 'Positive'

    else:
        return 'Unknown'

In [ ]:
philly_reviews[
    'rating_class'
] = philly_reviews[
    'review_stars'
].apply(classify_star_rating)

In [ ]:
philly_reviews[
    'rating_class'
].value_counts()

In [ ]:
(
    philly_reviews[
        'rating_class'
    ]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

In [ ]:
## Validate potentil dupiclate or missing data
print(
    'Duplicate review IDs:',
    philly_reviews[
        'review_id'
    ].duplicated().sum()
)

In [ ]:
print(
    'Missing business IDs:',
    philly_reviews[
        'business_id'
    ].isna().sum()
)

In [ ]:
print(
    'Missing user IDs:',
    philly_reviews[
        'user_id'
    ].isna().sum()
)

In [ ]:
print(
    'Invalid star ratings:',
    (
        ~philly_reviews[
            'review_stars'
        ].between(1, 5)
    ).sum()
)

In [ ]:
## Confirm all reviews are in the scope of our sample
print(
    'Reviews outside Philadelphia restaurant population:',
    (
        ~philly_reviews[
            'business_id'
        ].isin(philly_business_ids)
    ).sum()
)

In [ ]:
## Free full dataset from memory
print(
    f'Full review dataset memory: '
    f'{reviews.memory_usage(deep=True).sum() / 1024**2:,.2f} MB'
)

print(
    f'Philadelphia review dataset memory: '
    f'{philly_reviews.memory_usage(deep=True).sum() / 1024**2:,.2f} MB'
)

In [ ]:
del reviews

In [ ]:
print(
    f'Philadelphia restaurants: '
    f'{len(philly_business):,}'
)

print(
    f'Philadelphia reviews: '
    f'{len(philly_reviews):,}'
)

print(
    f'Restaurants with reviews: '
    f'{philly_reviews["business_id"].nunique():,}'
)

print(
    f'Unique reviewers: '
    f'{philly_reviews["user_id"].nunique():,}'
)

In [ ]:
reviews_per_restaurant[
    'observed_reviews'
].describe()

##### There is an uneven number of reviews per restaurant. The median is only 10 reviews even though the mean is 25.8, and the maximum is 316. That large mean– median difference tells us the distribution is strongly right-skewed: a relatively small group of restaurants accounts for a substantial share of reviews.

In [ ]:
thresholds = [1, 5, 10, 15, 20, 25, 30, 50]

threshold_results = []

for threshold in thresholds:

    restaurant_count = (
        reviews_per_restaurant[
            'observed_reviews'
        ] >= threshold
    ).sum()

    review_count = (
        reviews_per_restaurant.loc[
            reviews_per_restaurant[
                'observed_reviews'
            ] >= threshold,
            'observed_reviews'
        ].sum()
    )

    threshold_results.append({
        'Minimum_Reviews': threshold,
        'Restaurants_Retained': restaurant_count,
        'Percent_Restaurants_Retained':
            restaurant_count / len(reviews_per_restaurant) * 100,
        'Reviews_Retained': review_count,
        'Percent_Reviews_Retained':
            review_count / len(philly_reviews) * 100
    })

threshold_summary = pd.DataFrame(threshold_results)

threshold_summary[
    'Percent_Restaurants_Retained'
] = threshold_summary[
    'Percent_Restaurants_Retained'
].round(2)

threshold_summary[
    'Percent_Reviews_Retained'
] = threshold_summary[
    'Percent_Reviews_Retained'
].round(2)

threshold_summary

#### The Philadelphia restaurant population contained 484 businesses and 12,498 reviews; however, review volume was highly right-skewed across restaurants. Threshold sensitivity analysis was therefore conducted before establishing a minimum review requirement for restaurant-level analyses. A threshold of 10 observed reviews retained 243 restaurants (50.2% of the restaurant population) while preserving 11,569 reviews (92.6% of available review observations). This threshold was selected as a balance between restaurant-level measurement reliability and sufficient sample size for subsequent clustering, benchmarking, and predictive analyses. Importantly, the threshold is not applied globally: all eligible reviews remain available for review-level NLP analyses, and all 484 restaurants remain available for market-level exploratory analysis.

#### Going Forward:
* All 484 restaurants for market-level EDA, geographic analysis, ratings distributions, categories, business attributes, etc.

* All 12,498 usable reviews for NLP, sentiment analysis, topic modeling, text classification, and other review-level analyses.

* 243 restaurants with ≥10 reviews for analyses requiring reasonably stable restaurant-level NLP aggregates, such as clustering and benchmarking.

* For forecasting, I'll create a different eligibility criterion based on temporal coverage rather than number of reviews

In [ ]:
MIN_REVIEWS_PER_RESTAURANT = 10

In [ ]:
## Load Tip Data
tips = pd.read_json(
    tip_file_path
)

print(f'Total Yelp tips: {len(tips):,}')
print(f'Total columns: {tips.shape[1]}')

In [ ]:
tips.head()

In [ ]:
tips.info()

In [ ]:
tips.columns.tolist()

In [ ]:
## Filter to Philadephia Data

philly_tips = tips[
    tips['business_id'].isin(philly_business_ids)
].copy()

In [ ]:
print(f'All Yelp tips: {len(tips):,}')

print(
    f'Philadelphia restaurant tips: '
    f'{len(philly_tips):,}'
)

print(
    f'Restaurants with tips: '
    f'{philly_tips["business_id"].nunique():,}'
)

print(
    f'Unique tip users: '
    f'{philly_tips["user_id"].nunique():,}'
)

In [ ]:
tip_restaurant_coverage = (
    philly_tips['business_id'].nunique()
    / len(philly_business)
)

print(
    f'Restaurant tip coverage: '
    f'{tip_restaurant_coverage:.2%}'
)

#### These are strong results. Reviews remain our primary NLP source, tips become a complementary short-form NLP source

In [ ]:
## Check for missing values
print(
    'Missing business IDs:',
    philly_tips['business_id'].isna().sum()
)

print(
    'Missing user IDs:',
    philly_tips['user_id'].isna().sum()
)

print(
    'Missing tip text:',
    philly_tips['text'].isna().sum()
)

print(
    'Missing dates:',
    philly_tips['date'].isna().sum()
)

In [ ]:
empty_tip_text = (
    philly_tips['text']
    .fillna('')
    .str.strip()
    .eq('')
    .sum()
)

print(
    f'Empty tip text: '
    f'{empty_tip_text:,}'
)

In [ ]:
# Check for duplicates
print(
    'Fully duplicated rows:',
    philly_tips.duplicated().sum()
)

In [ ]:
## Check if any identical text was left
duplicate_tip_content = (
    philly_tips
    .duplicated(
        subset=[
            'user_id',
            'business_id',
            'text'
        ]
    )
    .sum()
)

print(
    f'Duplicate user-business-text combinations: '
    f'{duplicate_tip_content:,}'
)

In [ ]:
## Convert Dates
philly_tips['date'] = pd.to_datetime(
    philly_tips['date'],
    errors='coerce'
)

In [ ]:
print(
    'Earliest tip:',
    philly_tips['date'].min()
)

print(
    'Latest tip:',
    philly_tips['date'].max()
)

print(
    'Invalid/missing dates:',
    philly_tips['date'].isna().sum()
)

In [ ]:
philly_tips['year'] = (
    philly_tips['date']
    .dt.year
)

philly_tips['month'] = (
    philly_tips['date']
    .dt.month
)

philly_tips['year_month'] = (
    philly_tips['date']
    .dt.to_period('M')
)

philly_tips['day_of_week'] = (
    philly_tips['date']
    .dt.day_name()
)

In [ ]:
philly_tips[
    [
        'date',
        'year',
        'month',
        'year_month',
        'day_of_week'
    ]
].head()

In [ ]:
## Create text length features
philly_tips[
    'tip_character_count'
] = (
    philly_tips['text']
    .fillna('')
    .str.len()
)

In [ ]:
philly_tips[
    'tip_word_count'
] = (
    philly_tips['text']
    .fillna('')
    .str.split()
    .str.len()
)

In [ ]:
philly_tips[
    [
        'tip_character_count',
        'tip_word_count'
    ]
].describe()

In [ ]:
# Read some of the tips
philly_tips[
    [
        'text'
    ]
].sample(
    n=min(20, len(philly_tips)),
    random_state=42
)

In [ ]:
## Find tips per restaurant
tips_per_restaurant = (
    philly_tips
    .groupby('business_id')
    .size()
    .reset_index(
        name='observed_tips'
    )
)

In [ ]:
tips_per_restaurant[
    'observed_tips'
].describe()

In [ ]:
tips_per_restaurant.sort_values(
    'observed_tips',
    ascending=False
).head(20)

In [ ]:
tips_per_restaurant = (
    tips_per_restaurant
    .merge(
        philly_business[
            [
                'business_id',
                'name'
            ]
        ],
        on='business_id',
        how='left',
        validate='one_to_one'
    )
)

In [ ]:
tips_per_restaurant[
    [
        'name',
        'observed_tips'
    ]
].sort_values(
    'observed_tips',
    ascending=False
).head(20)

In [ ]:
## Examine Compliment Count

philly_tips[
    'compliment_count'
].describe()

In [ ]:
philly_tips[
    'compliment_count'
].value_counts().sort_index()

In [ ]:
## Indetify Tip Users
review_user_ids = set(
    philly_reviews[
        'user_id'
    ].dropna()
)

In [ ]:
tip_user_ids = set(
    philly_tips[
        'user_id'
    ].dropna()
)

In [ ]:
review_and_tip_users = (
    review_user_ids
    &
    tip_user_ids
)

print(
    f'Unique review users: '
    f'{len(review_user_ids):,}'
)

print(
    f'Unique tip users: '
    f'{len(tip_user_ids):,}'
)

print(
    f'Users appearing in both: '
    f'{len(review_and_tip_users):,}'
)

In [ ]:
print(
    f'Percent of tip users also appearing in reviews: '
    f'{len(review_and_tip_users) / len(tip_user_ids):.2%}'
)

In [ ]:
# Combine the two sources
relevant_user_ids = (
    review_user_ids
    |
    tip_user_ids
)

In [ ]:
print(
    f'Total relevant Yelp users: '
    f'{len(relevant_user_ids):,}'
)

In [ ]:
## Tip Summary
tip_summary = pd.DataFrame({
    'Metric': [
        'Philadelphia Restaurant Tips',
        'Restaurants with Tips',
        'Restaurant Tip Coverage',
        'Unique Tip Users'
    ],

    'Value': [
        f'{len(philly_tips):,}',
        f'{philly_tips["business_id"].nunique():,}',
        f'{philly_tips["business_id"].nunique() / len(philly_business):.2%}',
        f'{philly_tips["user_id"].nunique():,}'
    ]
})

tip_summary

In [ ]:
## Reviews vs. Tips analysis Later

In [ ]:
## Load check in data
checkins = pd.read_json(
    checkin_file_path
)

print(f'Total Yelp check-in records: {len(checkins):,}')
print(f'Total columns: {checkins.shape[1]}')

checkins.head()

In [ ]:
checkins.info()

In [ ]:
checkins.columns.tolist()

In [ ]:
## Filter to Philadelphia Data
philly_checkins = checkins[
    checkins['business_id'].isin(philly_business_ids)
].copy()

In [ ]:
print(f'All Yelp check-in records: {len(checkins):,}')

print(
    f'Philadelphia restaurant check-in records: '
    f'{len(philly_checkins):,}'
)

print(
    f'Restaurants with check-in data: '
    f'{philly_checkins["business_id"].nunique():,}'
)

In [ ]:
checkin_coverage = (
    philly_checkins['business_id'].nunique()
    / len(philly_business)
)

print(
    f'Restaurant check-in coverage: '
    f'{checkin_coverage:.2%}'
)

In [ ]:
## Inspect Timestamp Structure

pd.set_option(
    'display.max_colwidth',
    300
)

philly_checkins[
    ['business_id', 'date']
].head()

In [ ]:
## Convert to usable date structure
philly_checkins_long = (
    philly_checkins
    .assign(
        date=philly_checkins[
            'date'
        ].str.split(', ')
    )
    .explode('date')
    .reset_index(drop=True)
)

In [ ]:
philly_checkins_long[
    'date'
] = pd.to_datetime(
    philly_checkins_long[
        'date'
    ],
    errors='coerce'
)

In [ ]:
philly_checkins_long.head(10)

In [ ]:
print(
    f'Total Philadelphia restaurant check-in events: '
    f'{len(philly_checkins_long):,}'
)

In [ ]:
print(
    f'Restaurants represented: '
    f'{philly_checkins_long["business_id"].nunique():,}'
)

In [ ]:
print(
    'Earliest check-in:',
    philly_checkins_long['date'].min()
)

print(
    'Latest check-in:',
    philly_checkins_long['date'].max()
)

print(
    'Invalid/missing timestamps:',
    philly_checkins_long['date'].isna().sum()
)

In [ ]:
## Create Temporal features
philly_checkins_long[
    'year'
] = philly_checkins_long[
    'date'
].dt.year

philly_checkins_long[
    'month'
] = philly_checkins_long[
    'date'
].dt.month

philly_checkins_long[
    'year_month'
] = philly_checkins_long[
    'date'
].dt.to_period('M')

philly_checkins_long[
    'day_of_week'
] = philly_checkins_long[
    'date'
].dt.day_name()

philly_checkins_long[
    'hour'
] = philly_checkins_long[
    'date'
].dt.hour

In [ ]:
philly_checkins_long[
    [
        'business_id',
        'date',
        'year',
        'month',
        'year_month',
        'day_of_week',
        'hour'
    ]
].head()

In [ ]:
## Check ins per restaurant
checkins_per_restaurant = (
    philly_checkins_long
    .groupby('business_id')
    .size()
    .reset_index(
        name='observed_checkins'
    )
)

In [ ]:
checkins_per_restaurant[
    'observed_checkins'
].describe()

In [ ]:
checkins_per_restaurant = (
    checkins_per_restaurant
    .merge(
        philly_business[
            [
                'business_id',
                'name'
            ]
        ],
        on='business_id',
        how='left',
        validate='one_to_one'
    )
)

In [ ]:
checkins_per_restaurant[
    [
        'name',
        'observed_checkins'
    ]
].sort_values(
    'observed_checkins',
    ascending=False
).head(20)

In [ ]:
## Check in per year
checkins_by_year = (
    philly_checkins_long
    .groupby('year')
    .size()
    .reset_index(
        name='checkin_count'
    )
)

checkins_by_year

In [ ]:
restaurants_by_year = (
    philly_checkins_long
    .groupby('year')[
        'business_id'
    ]
    .nunique()
    .reset_index(
        name='active_restaurants'
    )
)

restaurants_by_year

In [ ]:
# Overall date range
print(
    'Earliest check-in:',
    philly_checkins_long['date'].min()
)

print(
    'Latest check-in:',
    philly_checkins_long['date'].max()
)

print(
    'Total check-in events:',
    f'{len(philly_checkins_long):,}'
)


# Annual check-in volume

checkins_by_year = (
    philly_checkins_long
    .groupby('year')
    .size()
    .reset_index(name='checkin_count')
)


# Number of restaurants represented each year

restaurants_by_year = (
    philly_checkins_long
    .groupby('year')['business_id']
    .nunique()
    .reset_index(name='active_restaurants')
)


# Combine the two tables


checkin_year_summary = (
    checkins_by_year
    .merge(
        restaurants_by_year,
        on='year',
        how='left'
    )
)


# Normalize activity by restaurant coverage
checkin_year_summary[
    'checkins_per_active_restaurant'
] = (
    checkin_year_summary['checkin_count']
    /
    checkin_year_summary['active_restaurants']
).round(2)


# Display results
checkin_year_summary

Yelp check-in records were filtered to the 484 Philadelphia restaurants included in the study. The original check-in data stored multiple timestamps within individual business records, so timestamps were separated to create an event-level dataset in which each row represents one observed Yelp check-in. The resulting dataset contained 122,247 check-in events across 466 restaurants, representing 96.28% of the restaurant population. Check-in activity ranged from January 2010 through January 2022 and was highly right-skewed across restaurants, with a median of 83 check-ins compared with a mean of 262.33 and a maximum of 3,287. Temporal features including year, month, day of week, hour, and year-month were derived for later analysis. Because Yelp check-ins represent voluntary platform engagement rather than verified restaurant visits, they are treated as a proxy for digital customer engagement rather than direct measures of customer traffic. Additionally, 2022 is identified as a partial year because the available data ends on January 18, 2022.

In [ ]:
## Create sample user set

relevant_user_ids = (
    review_user_ids
    |
    tip_user_ids
)

print(
    f'Review users: {len(review_user_ids):,}'
)

print(
    f'Tip users: {len(tip_user_ids):,}'
)

print(
    f'Total relevant unique users: '
    f'{len(relevant_user_ids):,}'
)

In [ ]:
## Load User data
users = pd.read_json(
    user_file_path
)

print(f'Total Yelp users: {len(users):,}')
print(f'Total columns: {users.shape[1]}')

In [ ]:
users.head()

In [ ]:
users.info()

In [ ]:
users.columns.tolist()

In [ ]:
## Filter to Philadelphia Users
philly_users = users[
    users['user_id'].isin(relevant_user_ids)
].copy()

print(
    f'All Yelp users: '
    f'{len(users):,}'
)

print(
    f'Relevant Philadelphia users: '
    f'{len(philly_users):,}'
)

In [ ]:
## Check how many people left reviews/tips are in the users dataset
matched_user_ids = set(
    philly_users['user_id']
)

user_match_rate = (
    len(matched_user_ids)
    /
    len(relevant_user_ids)
)

print(
    f'User match rate: '
    f'{user_match_rate:.2%}'
)

In [ ]:
## Check for duplicates
print(
    'Duplicate user IDs:',
    philly_users[
        'user_id'
    ].duplicated().sum()
)

In [ ]:
## Convert "Yelping Since" to date time
philly_users[
    'yelping_since'
] = pd.to_datetime(
    philly_users[
        'yelping_since'
    ],
    errors='coerce'
)

In [ ]:
## Check date range
print(
    'Earliest Yelp user start date:',
    philly_users[
        'yelping_since'
    ].min()
)

print(
    'Latest Yelp user start date:',
    philly_users[
        'yelping_since'
    ].max()
)

In [ ]:
## Check for missing dates
print(
    'Missing/invalid user start dates:',
    philly_users[
        'yelping_since'
    ].isna().sum()
)

In [ ]:
## Create Binary Flag for Elite Users
philly_users[
    'is_elite'
] = (
    philly_users[
        'elite'
    ]
    .fillna('')
    .str.strip()
    .ne('')
)

philly_users[
    'is_elite'
].value_counts()

In [ ]:
## Create Number of Elite Status years field
philly_users[
    'elite_years_count'
] = (
    philly_users[
        'elite'
    ]
    .fillna('')
    .apply(
        lambda x:
        0 if x.strip() == ''
        else len(x.split(','))
    )
)

In [ ]:
## Create number of friends field
philly_users[
    'friend_count_calculated'
] = (
    philly_users[
        'friends'
    ]
    .fillna('')
    .apply(
        lambda x:
        0 if x.strip() == ''
        else len(x.split(','))
    )
)

In [ ]:
## Look at numeric variables only
user_numeric_columns = [
    'review_count',
    'useful',
    'funny',
    'cool',
    'fans',
    'average_stars',
    'elite_years_count'
]

philly_users[
    user_numeric_columns
].describe().T

In [ ]:
## Check reviews per user
reviews_per_user = (
    philly_reviews
    .groupby(
        'user_id'
    )
    .size()
    .reset_index(
        name='philly_review_count'
    )
)

In [ ]:
reviews_per_user[
    'philly_review_count'
].describe()

In [ ]:
reviews_per_user.sort_values(
    'philly_review_count',
    ascending=False
).head(20)

In [ ]:
## Create a clean user table with relevant fields
user_features = philly_users[
    [
        'user_id',
        'review_count',
        'yelping_since',
        'useful',
        'funny',
        'cool',
        'fans',
        'average_stars',
        'is_elite',
        'elite_years_count'
    ]
].copy()

In [ ]:
## Standardize naming
user_features = (
    user_features
    .rename(
        columns={
            'review_count':
                'user_review_count',

            'useful':
                'user_useful_votes',

            'funny':
                'user_funny_votes',

            'cool':
                'user_cool_votes',

            'fans':
                'user_fans',

            'average_stars':
                'user_average_stars'
        }
    )
)

In [ ]:
## Merge Data
review_master = (
    philly_reviews
    .merge(
        user_features,
        on='user_id',
        how='left',
        validate='many_to_one'
    )
)

In [ ]:
review_master.head()

I focused on loading, validating, filtering, and profiling the five Yelp JSON datasets to establish a consistent analytical population for the project. After confirming the files were stored as standard JSON arrays, the business data was filtered to 484 Philadelphia restaurants, whose business_id values were then used to filter all other datasets. This resulted in 12,498 reviews from 9,963 unique reviewers with 100% restaurant coverage, 3,173 tips covering 368 restaurants (76.03%), and 122,247 individual check-in events covering 466 restaurants (96.28%). The check-in data required additional restructuring because multiple timestamps were stored within each business record; these were separated into individual event-level observations spanning January 2010 through January 2022. The user dataset was also restricted to users appearing in the Philadelphia review or tip data, ensuring that only information relevant to the defined restaurant market was retained.

Initial profiling was conducted to understand data coverage, distributions, and potential limitations before formal cleaning or analysis. Review activity was highly uneven across restaurants, so a sensitivity analysis established 10 reviews as a reasonable threshold for later restaurant-level analyses, retaining 243 restaurants and 92.6% of all reviews; however, all 484 restaurants and 12,498 reviews remain available for analyses that do not require this threshold. Reviews were designated as the primary unstructured customer-voice source, while tips were retained as complementary short-form text that can later be compared with formal reviews. Check-ins were defined as a proxy for Yelp digital customer engagement rather than actual restaurant traffic, and 2022 was identified as a partial year.

## Check Data Quality

In [ ]:
print(
    'Missing categories:',
    philly_business['categories'].isna().sum()
)

print(
    'Unique category strings:',
    philly_business['categories'].nunique()
)

In [ ]:
## Check number of categories per each restaurant
philly_business['category_count'] = (
    philly_business['categories']
    .fillna('')
    .apply(
        lambda x:
        len([
            category.strip()
            for category in x.split(',')
            if category.strip()
        ])
    )
)

philly_business[
    'category_count'
].describe()

In [ ]:
## Convert Categories into lists
philly_business[
    'category_list'
] = (
    philly_business['categories']
    .fillna('')
    .apply(
        lambda x: [
            category.strip()
            for category in x.split(',')
            if category.strip()
        ]
    )
)

In [ ]:
philly_business[
    [
        'name',
        'categories',
        'category_list'
    ]
].head()

In [ ]:
## Create long format category table
restaurant_categories = (
    philly_business[
        [
            'business_id',
            'name',
            'category_list'
        ]
    ]
    .explode('category_list')
    .rename(
        columns={
            'category_list': 'category'
        }
    )
    .reset_index(drop=True)
)

In [ ]:
restaurant_categories.head(15)

In [ ]:
## Look at yelp category vocabulary
category_frequency = (
    restaurant_categories[
        'category'
    ]
    .value_counts()
    .reset_index()
)

category_frequency.columns = [
    'category',
    'restaurant_count'
]

category_frequency.head(50)

In [ ]:
print(
    'Unique Yelp categories:',
    restaurant_categories[
        'category'
    ].nunique()
)

In [ ]:
## Remove Generic Categories for Cuisine classification
generic_categories = {
    'Restaurants',
    'Food',
    'Nightlife',
    'Bars',
    'Event Planning & Services'
}

In [ ]:
restaurant_categories[
    'is_generic_category'
] = (
    restaurant_categories[
        'category'
    ].isin(
        generic_categories
    )
)

In [ ]:
## Create Restaurant Category flags
def has_category(category_list, category):
    """
    Return True if a Yelp category appears
    in a restaurant's category list.
    """
    return category in category_list

In [ ]:
philly_business[
    'is_bar'
] = philly_business[
    'category_list'
].apply(
    lambda x:
    has_category(x, 'Bars')
)

philly_business[
    'is_coffee_tea'
] = philly_business[
    'category_list'
].apply(
    lambda x:
    has_category(x, 'Coffee & Tea')
)

philly_business[
    'is_bakery'
] = philly_business[
    'category_list'
].apply(
    lambda x:
    has_category(x, 'Bakeries')
)

philly_business[
    'is_fast_food'
] = philly_business[
    'category_list'
].apply(
    lambda x:
    has_category(x, 'Fast Food')
)

philly_business[
    'is_food_truck'
] = philly_business[
    'category_list'
].apply(
    lambda x:
    has_category(x, 'Food Trucks')
)

philly_business[
    'is_breakfast_brunch'
] = philly_business[
    'category_list'
].apply(
    lambda x:
    has_category(x, 'Breakfast & Brunch')
)

In [ ]:
philly_business[
    [
        'is_bar',
        'is_coffee_tea',
        'is_bakery',
        'is_fast_food',
        'is_food_truck',
        'is_breakfast_brunch'
    ]
].sum()

In [ ]:
# Combine Cuisines into broader categories
cuisine_mapping = {

    # American
    'American (New)': 'American',
    'American (Traditional)': 'American',
    'Southern': 'American',
    'Soul Food': 'American',
    'Cajun/Creole': 'American',
    'Comfort Food': 'American',

    # Italian
    'Italian': 'Italian',

    # Chinese
    'Chinese': 'Chinese',
    'Szechuan': 'Chinese',
    'Cantonese': 'Chinese',
    'Dim Sum': 'Chinese',

    # Japanese
    'Japanese': 'Japanese',
    'Sushi Bars': 'Japanese',
    'Ramen': 'Japanese',

    # Mexican / Tex-Mex
    'Mexican': 'Mexican / Tex-Mex',
    'Tacos': 'Mexican / Tex-Mex',
    'Tex-Mex': 'Mexican / Tex-Mex',

    # Southeast Asian
    'Thai': 'Southeast Asian',
    'Vietnamese': 'Southeast Asian',

    # South Asian
    'Indian': 'South Asian',
    'Pakistani': 'South Asian',

    # Mediterranean / Middle Eastern
    'Mediterranean': 'Mediterranean / Middle Eastern',
    'Middle Eastern': 'Mediterranean / Middle Eastern',
    'Greek': 'Mediterranean / Middle Eastern',
    'Falafel': 'Mediterranean / Middle Eastern',

    # Latin American / Caribbean
    'Latin American': 'Latin / Caribbean',
    'Caribbean': 'Latin / Caribbean',
    'Cuban': 'Latin / Caribbean',

    # Korean
    'Korean': 'Korean',

    # Mixed / Fusion
    'Asian Fusion': 'Other / Mixed',

    # Other cuisines
    'Moroccan': 'Other / Mixed',
    'Irish': 'Other / Mixed'
}

In [ ]:
# Allow for multiple cuisine groups
def get_cuisine_groups(category_list):
    """
    Identify all broad cuisine groups represented
    in a restaurant's Yelp category list.
    """

    cuisine_groups = []

    for category in category_list:

        if category in cuisine_mapping:

            cuisine_group = cuisine_mapping[
                category
            ]

            if cuisine_group not in cuisine_groups:
                cuisine_groups.append(
                    cuisine_group
                )

    return cuisine_groups

In [ ]:
philly_business[
    'cuisine_groups'
] = (
    philly_business[
        'category_list'
    ].apply(
        get_cuisine_groups
    )
)

In [ ]:
philly_business[
    [
        'name',
        'categories',
        'cuisine_groups'
    ]
].head(20)

In [ ]:
## Create Cuisine Binary Features
cuisine_groups = [
    'American',
    'Italian',
    'Chinese',
    'Japanese',
    'Mexican / Tex-Mex',
    'Southeast Asian',
    'South Asian',
    'Mediterranean / Middle Eastern',
    'Latin / Caribbean',
    'Korean',
    'Other / Mixed'
]

In [ ]:
for cuisine in cuisine_groups:

    column_name = (
        'cuisine_'
        + cuisine
        .lower()
        .replace(' / ', '_')
        .replace(' ', '_')
    )

    philly_business[
        column_name
    ] = (
        philly_business[
            'cuisine_groups'
        ].apply(
            lambda x:
            cuisine in x
        )
    )

In [ ]:
[
    column
    for column in philly_business.columns
    if column.startswith('cuisine_')
]

In [ ]:
cuisine_flag_columns = [
    column
    for column in philly_business.columns
    if column.startswith('cuisine_')
    and column != 'cuisine_groups'
]

philly_business[
    cuisine_flag_columns
].sum().sort_values(
    ascending=False
)

In [ ]:
# Now map the food sepcialty restaurants rather than cuisines
food_specialties = {

    'pizza': 'Pizza',

    'sandwiches': 'Sandwiches',

    'seafood': 'Seafood',

    'burgers': 'Burgers',

    'salad': 'Salad',

    'chicken_wings': 'Chicken Wings',

    'cheesesteaks': 'Cheesesteaks',

    'desserts': 'Desserts',

    'vegetarian': 'Vegetarian',

    'vegan': 'Vegan',

    'halal': 'Halal',

    'soup': 'Soup',

    'noodles': 'Noodles',

    'barbeque': 'Barbeque',

    'chicken': 'Chicken Shop'
}

In [ ]:
for feature_name, yelp_category in food_specialties.items():

    philly_business[
        f'food_{feature_name}'
    ] = (
        philly_business[
            'category_list'
        ].apply(
            lambda x:
            yelp_category in x
        )
    )

In [ ]:
food_flag_columns = [
    column
    for column in philly_business.columns
    if column.startswith('food_')
]

philly_business[
    food_flag_columns
].sum().sort_values(
    ascending=False
)

In [ ]:
## Formart Restaurant type
restaurant_formats = {

    'breakfast_brunch':
        'Breakfast & Brunch',

    'coffee_tea':
        'Coffee & Tea',

    'deli':
        'Delis',

    'fast_food':
        'Fast Food',

    'cafe':
        'Cafes',

    'bakery':
        'Bakeries',

    'food_truck':
        'Food Trucks',

    'food_stand':
        'Food Stands',

    'diner':
        'Diners',

    'buffet':
        'Buffets',

    'street_vendor':
        'Street Vendors',

    'caterer':
        'Caterers'
}

In [ ]:
for feature_name, yelp_category in restaurant_formats.items():

    philly_business[
        f'format_{feature_name}'
    ] = (
        philly_business[
            'category_list'
        ].apply(
            lambda x:
            yelp_category in x
        )
    )

In [ ]:
format_columns = [
    column
    for column in philly_business.columns
    if column.startswith('format_')
]

philly_business[
    format_columns
].sum().sort_values(
    ascending=False
)

In [ ]:
## Flag the nightlife restaurants
nightlife_categories = {

    'bar':
        'Bars',

    'pub':
        'Pubs',

    'cocktail_bar':
        'Cocktail Bars',

    'lounge':
        'Lounges',

    'sports_bar':
        'Sports Bars',

    'wine_bar':
        'Wine Bars',

    'gastropub':
        'Gastropubs',

    'karaoke':
        'Karaoke',

    'brewery':
        'Breweries',

    'beer_bar':
        'Beer Bar'
}

In [ ]:
for feature_name, yelp_category in nightlife_categories.items():

    philly_business[
        f'nightlife_{feature_name}'
    ] = (
        philly_business[
            'category_list'
        ].apply(
            lambda x:
            yelp_category in x
        )
    )

In [ ]:
## Measure Cuisine Coverage
philly_business[
    'has_cuisine_classification'
] = (
    philly_business[
        'cuisine_groups'
    ].apply(
        lambda x:
        len(x) > 0
    )
)

In [ ]:
classified_count = (
    philly_business[
        'has_cuisine_classification'
    ].sum()
)

classified_percent = (
    classified_count
    /
    len(philly_business)
    * 100
)

print(
    f'Restaurants with cuisine classification: '
    f'{classified_count:,}'
)

print(
    f'Cuisine classification coverage: '
    f'{classified_percent:.2f}%'
)

In [ ]:
## Looks at unclassified restaurants
unclassified_restaurants = (
    philly_business[
        ~philly_business[
            'has_cuisine_classification'
        ]
    ][
        [
            'name',
            'categories'
        ]
    ]
    .copy()
)

In [ ]:
print(
    f'Unclassified restaurants: '
    f'{len(unclassified_restaurants):,}'
)

unclassified_restaurants.head(50)

In [ ]:
## Look at categories of unclassified restaurants

unclassified_category_frequency = (
    philly_business.loc[
        ~philly_business['has_cuisine_classification'],
        ['business_id', 'category_list']
    ]
    .explode('category_list')
    .rename(
        columns={
            'category_list': 'category'
        }
    )
    ['category']
    .value_counts()
    .reset_index()
)

unclassified_category_frequency.columns = [
    'category',
    'restaurant_count'
]

unclassified_category_frequency.head(100)

In [ ]:
## After further review add more mapping categories

additional_cuisine_mapping = {

    # European
    'French': 'European',
    'Spanish': 'European',
    'Portuguese': 'European',

    # Additional Latin / Caribbean
    'Peruvian': 'Latin / Caribbean',
    'Colombian': 'Latin / Caribbean',
    'Puerto Rican': 'Latin / Caribbean',
    'Dominican': 'Latin / Caribbean',

    # Additional Southeast Asian
    'Malaysian': 'Southeast Asian',
    'Cambodian': 'Southeast Asian',
    'Indonesian': 'Southeast Asian',

    # Additional South Asian
    'Bangladeshi': 'South Asian',
    'Sri Lankan': 'South Asian',

    # Other clearly defined cuisines
    'African': 'Other / Mixed',
    'Ethiopian': 'Other / Mixed'
}

In [ ]:
## Update Dictionary
cuisine_mapping.update(
    additional_cuisine_mapping
)

In [ ]:
if 'European' not in cuisine_groups:
    cuisine_groups.append('European')

In [ ]:
## Re run cuisine assignment
philly_business[
    'cuisine_groups'
] = (
    philly_business[
        'category_list'
    ].apply(
        get_cuisine_groups
    )
)

In [ ]:
for cuisine in cuisine_groups:

    column_name = (
        'cuisine_'
        + cuisine
        .lower()
        .replace(' / ', '_')
        .replace(' ', '_')
    )

    philly_business[
        column_name
    ] = (
        philly_business[
            'cuisine_groups'
        ].apply(
            lambda x:
            cuisine in x
        )
    )

In [ ]:
## Re calculate coverage

philly_business[
    'has_cuisine_classification'
] = (
    philly_business[
        'cuisine_groups'
    ].apply(
        lambda x:
        len(x) > 0
    )
)

classified_count = (
    philly_business[
        'has_cuisine_classification'
    ].sum()
)

classified_percent = (
    classified_count
    / len(philly_business)
    * 100
)

print(
    f'Restaurants with cuisine classification: '
    f'{classified_count:,}'
)

print(
    f'Cuisine classification coverage: '
    f'{classified_percent:.2f}%'
)

In [ ]:
## Create cuisine availibilty label
philly_business[
    'cuisine_classification_status'
] = np.where(
    philly_business[
        'has_cuisine_classification'
    ],
    'Explicit Cuisine Identified',
    'No Explicit Cuisine'
)

In [ ]:
## Fix hyphen
for cuisine in cuisine_groups:

    column_name = (
        'cuisine_'
        + cuisine
        .lower()
        .replace(' / ', '_')
        .replace(' ', '_')
        .replace('-', '_')
    )

306 of 484 Philadelphia restaurants (63.22%) contain at least one explicit cultural cuisine label in their Yelp categories. The remaining restaurants are primarily described through food specialties, service formats, or business types such as pizza, sandwiches, fast food, bakeries, cafés, breakfast restaurants, seafood, and cheesesteaks. Rather than forcing these businesses into arbitrary cultural cuisine groups, the project preserves Yelp's multi-label structure through separate cuisine, food-specialty, restaurant-format, and nightlife features

In [ ]:
unclassified_category_frequency.head(100)

In [ ]:
additional_cuisine_mapping = {

    # European
    'French': 'European',
    'Polish': 'European',

    # African
    'African': 'African',
    'Senegalese': 'African',
    'Ethiopian': 'African',

    # Additional Southeast Asian
    'Filipino': 'Southeast Asian',
    'Cambodian': 'Southeast Asian',
    'Indonesian': 'Southeast Asian'
}

cuisine_mapping.update(
    additional_cuisine_mapping
)

In [ ]:
if 'European' not in cuisine_groups:
    cuisine_groups.append('European')

if 'African' not in cuisine_groups:
    cuisine_groups.append('African')

In [ ]:
philly_business[
    'cuisine_groups'
] = (
    philly_business[
        'category_list'
    ].apply(
        get_cuisine_groups
    )
)

In [ ]:
for cuisine in cuisine_groups:

    column_name = (
        'cuisine_'
        + cuisine
        .lower()
        .replace(' / ', '_')
        .replace(' ', '_')
        .replace('-', '_')
    )

    philly_business[
        column_name
    ] = (
        philly_business[
            'cuisine_groups'
        ].apply(
            lambda x:
            cuisine in x
        )
    )

In [ ]:
philly_business[
    'has_cuisine_classification'
] = (
    philly_business[
        'cuisine_groups'
    ].apply(
        lambda x:
        len(x) > 0
    )
)

classified_count = (
    philly_business[
        'has_cuisine_classification'
    ].sum()
)

classified_percent = (
    classified_count
    / len(philly_business)
    * 100
)

print(
    f'Restaurants with cuisine classification: '
    f'{classified_count:,}'
)

print(
    f'Cuisine classification coverage: '
    f'{classified_percent:.2f}%'
)

In [ ]:
philly_business[
    'cuisine_classification_status'
] = np.where(
    philly_business[
        'has_cuisine_classification'
    ],
    'Explicit Cuisine Identified',
    'No Explicit Cuisine'
)

309 of 484 Philadelphia restaurants (63.84%) contain at least one explicit cultural cuisine label in their Yelp categories. The remaining restaurants are primarily described through food specialties, service formats, or business types such as pizza, sandwiches, fast food, bakeries, cafés, breakfast restaurants, seafood, and cheesesteaks. Rather than forcing these businesses into arbitrary cultural cuisine groups, the project preserves Yelp's multi-label structure through separate cuisine, food-specialty, restaurant-format, and nightlife features

In [ ]:
## Understand Attribute Frequency
from collections import Counter

attribute_counter = Counter()

for attributes in philly_business['attributes'].dropna():

    if isinstance(attributes, dict):

        attribute_counter.update(
            attributes.keys()
        )


attribute_frequency = (
    pd.DataFrame(
        attribute_counter.items(),
        columns=[
            'attribute',
            'restaurant_count'
        ]
    )
    .sort_values(
        'restaurant_count',
        ascending=False
    )
    .reset_index(drop=True)
)

attribute_frequency[
    'coverage_percent'
] = (
    attribute_frequency[
        'restaurant_count'
    ]
    / len(philly_business)
    * 100
).round(2)

attribute_frequency

In [ ]:
## Attribute Extraction Function
def get_attribute(attributes, key):
    """
    Extract a specific attribute from Yelp's nested
    attributes dictionary.

    Parameters
    ----------
    attributes : dict
        Yelp attributes dictionary for a restaurant.

    key : str
        Attribute to retrieve.

    Returns
    -------
    Attribute value if available; otherwise None.
    """

    if not isinstance(attributes, dict):
        return None

    return attributes.get(key, None)

In [ ]:
# Test attribute extraction using delivery

delivery_raw = (
    philly_business['attributes']
    .apply(
        lambda x:
        get_attribute(
            x,
            'RestaurantsDelivery'
        )
    )
)

delivery_raw.value_counts(
    dropna=False
)

In [ ]:
## Clean Boolean Values

def clean_yelp_boolean(value):
    """
    Standardize Yelp boolean-like attribute values.

    Converts:
        'True'  -> True
        'False' -> False
        'None'  -> pd.NA
        None    -> pd.NA

    Missing information is preserved rather than
    incorrectly interpreted as False.
    """

    # Handle actual Python missing values
    if value is None or pd.isna(value):
        return pd.NA

    # Convert to standardized lowercase string
    value = str(value).strip().lower()

    # Convert known boolean values
    if value == 'true':
        return True

    if value == 'false':
        return False

    # Handle Yelp's string representation of missing values
    if value in ['none', 'nan', '']:
        return pd.NA

    # Anything unexpected is treated as missing
    return pd.NA

In [ ]:
# Apply Boolean cleaning function
delivery_clean = (
    delivery_raw
    .apply(clean_yelp_boolean)
    .astype('boolean')
)

In [ ]:
delivery_clean.value_counts(
    dropna=False
)

In [ ]:
delivery_validation = pd.DataFrame({
    'raw_value': delivery_raw,
    'clean_value': delivery_clean
})

delivery_validation.value_counts(
    dropna=False
)

In [ ]:
delivery_clean.value_counts(dropna=False)

In [ ]:
## Extract Boolean Attributes

boolean_attributes = {

    # Service options
    'offers_delivery':
        'RestaurantsDelivery',

    'offers_takeout':
        'RestaurantsTakeOut',

    'takes_reservations':
        'RestaurantsReservations',

    'table_service':
        'RestaurantsTableService',

    'caters':
        'Caters',

    # Customer / dining experience
    'good_for_kids':
        'GoodForKids',

    'good_for_groups':
        'RestaurantsGoodForGroups',

    'outdoor_seating':
        'OutdoorSeating',

    'has_tv':
        'HasTV',

    'happy_hour':
        'HappyHour',

    # Accessibility / amenities
    'accepts_credit_cards':
        'BusinessAcceptsCreditCards',

    'bike_parking':
        'BikeParking',

    'wheelchair_accessible':
        'WheelchairAccessible',

    'dogs_allowed':
        'DogsAllowed'
}

In [ ]:
## Extract and clean
for new_column, yelp_attribute in boolean_attributes.items():

    philly_business[new_column] = (
        philly_business['attributes']
        .apply(
            lambda x:
            get_attribute(
                x,
                yelp_attribute
            )
        )
        .apply(clean_yelp_boolean)
        .astype('boolean')
    )

In [ ]:
## Check
boolean_feature_columns = list(
    boolean_attributes.keys()
)

philly_business[
    boolean_feature_columns
].head()

In [ ]:
boolean_attribute_summary = []

for column in boolean_feature_columns:

    available_count = (
        philly_business[column]
        .notna()
        .sum()
    )

    missing_count = (
        philly_business[column]
        .isna()
        .sum()
    )

    true_count = (
        philly_business[column]
        .eq(True)
        .sum()
    )

    false_count = (
        philly_business[column]
        .eq(False)
        .sum()
    )

    boolean_attribute_summary.append(
        {
            'feature':
                column,

            'available_count':
                available_count,

            'coverage_percent':
                round(
                    available_count
                    / len(philly_business)
                    * 100,
                    2
                ),

            'true_count':
                true_count,

            'false_count':
                false_count,

            'missing_count':
                missing_count
        }
    )


boolean_attribute_summary = (
    pd.DataFrame(
        boolean_attribute_summary
    )
    .sort_values(
        'coverage_percent',
        ascending=False
    )
    .reset_index(drop=True)
)

boolean_attribute_summary

In [ ]:
## Look at price range
price_raw = (
    philly_business['attributes']
    .apply(
        lambda x:
        get_attribute(
            x,
            'RestaurantsPriceRange2'
        )
    )
)

price_raw.value_counts(
    dropna=False
).sort_index()

In [ ]:
## Clean Price Range
philly_business['price_range'] = (
    pd.to_numeric(
        price_raw,
        errors='coerce'
    )
    .astype('Int64')
)

In [ ]:
philly_business[
    'price_range'
].value_counts(
    dropna=False
).sort_index()

In [ ]:
## Create readable price labels

price_labels = {
    1: '$',
    2: '$$',
    3: '$$$',
    4: '$$$$'
}

philly_business[
    'price_category'
] = (
    philly_business[
        'price_range'
    ].map(
        price_labels
    )
)

philly_business[
    [
        'price_range',
        'price_category'
    ]
].value_counts(
    dropna=False
).sort_index()

Yelp's RestaurantsPriceRange2 attribute was converted from a semi-structured string representation to a nullable integer variable ranging from 1 to 4. Valid price information was available for 466 of 484 restaurants (96.28%), while 18 restaurants with unavailable price information were retained as missing rather than imputed. A corresponding categorical $–$$$$ label was also created for reporting and visualization.

In [ ]:
## Look at other categorical attributes
categorical_attributes = {
    'alcohol': 'Alcohol',
    'wifi': 'WiFi',
    'noise_level': 'NoiseLevel',
    'attire': 'RestaurantsAttire'
}


for feature_name, yelp_attribute in categorical_attributes.items():

    # Extract raw values from Yelp's attributes dictionary
    raw_values = (
        philly_business['attributes']
        .apply(
            lambda x:
            get_attribute(
                x,
                yelp_attribute
            )
        )
    )

    print(
        raw_values.value_counts(
            dropna=False
        )
    )

    print()

In [ ]:
## Clean Categorical Attributes

def clean_yelp_category(value):
    """
    Standardize Yelp categorical attribute values.

    Examples:
        u'free'       -> free
        'free'        -> free
        u'full_bar'   -> full_bar
        'casual'      -> casual
        None          -> pd.NA

    Parameters
    ----------
    value : object
        Raw Yelp attribute value.

    Returns
    -------
    Cleaned lowercase string or pd.NA.
    """

    # Handle actual missing values
    if value is None or pd.isna(value):
        return pd.NA

    # Convert to string and remove whitespace
    value = str(value).strip()

    # Remove Python unicode prefix:
    # u'free' -> 'free'
    if value.startswith("u'") or value.startswith('u"'):
        value = value[1:]

    # Remove surrounding quotation marks
    value = value.strip("'\"")

    # Standardize case
    value = value.lower()

    # Handle empty strings
    if value == '':
        return pd.NA

    return value

In [ ]:
categorical_attributes = {
    'alcohol': 'Alcohol',
    'wifi': 'WiFi',
    'noise_level': 'NoiseLevel',
    'attire': 'RestaurantsAttire'
}

In [ ]:
for new_column, yelp_attribute in categorical_attributes.items():

    philly_business[new_column] = (
        philly_business['attributes']
        .apply(
            lambda x:
            get_attribute(
                x,
                yelp_attribute
            )
        )
        .apply(clean_yelp_category)
    )

In [ ]:
## Validate
for column in categorical_attributes.keys():
    print(
        philly_business[column]
        .value_counts(dropna=False)
    )

    print()

In [ ]:
## Create Summary Table
categorical_attribute_summary = []

for column in categorical_attributes.keys():

    available_count = (
        philly_business[column]
        .notna()
        .sum()
    )

    missing_count = (
        philly_business[column]
        .isna()
        .sum()
    )

    unique_categories = (
        philly_business[column]
        .nunique()
    )

    categorical_attribute_summary.append(
        {
            'feature':
                column,

            'available_count':
                available_count,

            'coverage_percent':
                round(
                    available_count
                    / len(philly_business)
                    * 100,
                    2
                ),

            'unique_categories':
                unique_categories,

            'missing_count':
                missing_count
        }
    )


categorical_attribute_summary = pd.DataFrame(
    categorical_attribute_summary
)

categorical_attribute_summary

In [ ]:
## Inspect Buisness Parking Values
parking_raw = (
    philly_business['attributes']
    .apply(
        lambda x:
        get_attribute(
            x,
            'BusinessParking'
        )
    )
)

print(
    "Restaurants with BusinessParking key:",
    parking_raw.notna().sum()
)

print(
    "Restaurants without BusinessParking key:",
    parking_raw.isna().sum()
)

parking_raw.head(10)

In [ ]:
## Parse through nested dictionairy

import ast


def parse_nested_attribute(value):
    """
    Safely convert Yelp nested attribute strings
    into Python dictionaries.

    Examples
    --------
    "{'garage': False, 'street': True}"
        ->
    {'garage': False, 'street': True}

    Missing or invalid values return an empty dictionary.
    """

    # If already a dictionary, return as-is
    if isinstance(value, dict):
        return value

    # Handle actual missing values
    if value is None or pd.isna(value):
        return {}

    # Yelp may sometimes store the word None as text
    if str(value).strip().lower() == 'none':
        return {}

    # Safely parse string representation
    try:

        parsed_value = ast.literal_eval(
            str(value)
        )

        if isinstance(
            parsed_value,
            dict
        ):
            return parsed_value

    except (
        ValueError,
        SyntaxError
    ):
        pass

    return {}

In [ ]:
## Get one example
sample_parking = (
    parking_raw
    .dropna()
    .iloc[0]
)

print("RAW PARKING VALUE:")
print(sample_parking)

print("\nPARSED PARKING VALUE:")
print(
    parse_nested_attribute(
        sample_parking
    )
)

In [ ]:
## Go through all record
philly_business[
    'parking_dict'
] = (
    parking_raw
    .apply(
        parse_nested_attribute
    )
)

philly_business[
    'parking_dict'
].head()

In [ ]:
## Extract Parking Features
parking_features = [
    'garage',
    'street',
    'validated',
    'lot',
    'valet'
]


for parking_type in parking_features:

    philly_business[
        f'parking_{parking_type}'
    ] = (
        philly_business[
            'parking_dict'
        ]
        .apply(
            lambda x:
            x.get(
                parking_type,
                pd.NA
            )
        )
        .apply(
            clean_yelp_boolean
        )
        .astype('boolean')
    )

In [ ]:
## Parking Summary Table

parking_feature_columns = [
    f'parking_{x}'
    for x in parking_features
]

parking_summary = []

for column in parking_feature_columns:

    available_count = (
        philly_business[column]
        .notna()
        .sum()
    )

    true_count = (
        philly_business[column]
        .eq(True)
        .sum()
    )

    false_count = (
        philly_business[column]
        .eq(False)
        .sum()
    )

    missing_count = (
        philly_business[column]
        .isna()
        .sum()
    )

    parking_summary.append(
        {
            'feature':
                column,

            'available_count':
                available_count,

            'coverage_percent':
                round(
                    available_count
                    / len(philly_business)
                    * 100,
                    2
                ),

            'true_count':
                true_count,

            'false_count':
                false_count,

            'missing_count':
                missing_count
        }
    )


parking_summary = (
    pd.DataFrame(
        parking_summary
    )
    .sort_values(
        'coverage_percent',
        ascending=False
    )
    .reset_index(drop=True)
)

parking_summary

Yelp's BusinessParking attribute contained a second-level dictionary embedded as a string within the primary attributes object. These values were safely parsed using Python's ast.literal_eval() and decomposed into garage, street, lot, validated, and valet parking indicators. Missing subattributes were retained separately from explicit False values to avoid interpreting unavailable information as the absence of a parking option

In [ ]:
## Look at ambiance category
ambience_raw = (
    philly_business['attributes']
    .apply(
        lambda x:
        get_attribute(
            x,
            'Ambience'
        )
    )
)

print(
    "Restaurants with Ambience key:",
    ambience_raw.notna().sum()
)

print(
    "Restaurants without Ambience key:",
    ambience_raw.isna().sum()
)

In [ ]:
philly_business[
    'ambience_dict'
] = (
    ambience_raw
    .apply(
        parse_nested_attribute
    )
)

In [ ]:
ambience_keys = set()

for ambience in philly_business[
    'ambience_dict'
]:

    if isinstance(
        ambience,
        dict
    ):
        ambience_keys.update(
            ambience.keys()
        )

print(
    "Ambience sub-attributes:"
)

print(
    sorted(ambience_keys)
)

In [ ]:
ambience_features = [
    'casual',
    'classy',
    'divey',
    'hipster',
    'intimate',
    'romantic',
    'touristy',
    'trendy',
    'upscale'
]

In [ ]:
for ambience_type in ambience_features:

    philly_business[
        f'ambience_{ambience_type}'
    ] = (
        philly_business[
            'ambience_dict'
        ]
        .apply(
            lambda x:
            x.get(
                ambience_type,
                pd.NA
            )
        )
        .apply(
            clean_yelp_boolean
        )
        .astype('boolean')
    )

In [ ]:
ambience_feature_columns = [
    f'ambience_{x}'
    for x in ambience_features
]

ambience_summary = []

for column in ambience_feature_columns:

    available_count = (
        philly_business[
            column
        ].notna().sum()
    )

    true_count = (
        philly_business[
            column
        ].eq(True).sum()
    )

    false_count = (
        philly_business[
            column
        ].eq(False).sum()
    )

    missing_count = (
        philly_business[
            column
        ].isna().sum()
    )

    ambience_summary.append(
        {
            'feature':
                column,

            'available_count':
                available_count,

            'coverage_percent':
                round(
                    available_count
                    / len(philly_business)
                    * 100,
                    2
                ),

            'true_count':
                true_count,

            'false_count':
                false_count,

            'missing_count':
                missing_count
        }
    )


ambience_summary = (
    pd.DataFrame(
        ambience_summary
    )
    .sort_values(
        'coverage_percent',
        ascending=False
    )
    .reset_index(drop=True)
)

ambience_summary

In [ ]:
## Look at good for meal attribute
meal_raw = (
    philly_business['attributes']
    .apply(
        lambda x:
        get_attribute(
            x,
            'GoodForMeal'
        )
    )
)

print(
    "Restaurants with GoodForMeal key:",
    meal_raw.notna().sum()
)

print(
    "Restaurants without GoodForMeal key:",
    meal_raw.isna().sum()
)

In [ ]:
philly_business['meal_dict'] = (
    meal_raw
    .apply(parse_nested_attribute)
)

In [ ]:
meal_keys = set()

for meal in philly_business['meal_dict']:

    if isinstance(meal, dict):
        meal_keys.update(
            meal.keys()
        )

print(
    "GoodForMeal sub-attributes:",
    sorted(meal_keys)
)

In [ ]:
## Extract Features
meal_features = [
    'breakfast',
    'brunch',
    'lunch',
    'dinner',
    'latenight',
    'dessert'
]


for meal_type in meal_features:

    philly_business[
        f'meal_{meal_type}'
    ] = (
        philly_business['meal_dict']
        .apply(
            lambda x:
            x.get(
                meal_type,
                pd.NA
            )
        )
        .apply(clean_yelp_boolean)
        .astype('boolean')
    )

In [ ]:
## Create Summary Table
meal_feature_columns = [
    f'meal_{x}'
    for x in meal_features
]

meal_summary = []

for column in meal_feature_columns:

    available_count = (
        philly_business[column]
        .notna()
        .sum()
    )

    true_count = (
        philly_business[column]
        .eq(True)
        .sum()
    )

    false_count = (
        philly_business[column]
        .eq(False)
        .sum()
    )

    missing_count = (
        philly_business[column]
        .isna()
        .sum()
    )

    meal_summary.append(
        {
            'feature': column,

            'available_count':
                available_count,

            'coverage_percent':
                round(
                    available_count
                    / len(philly_business)
                    * 100,
                    2
                ),

            'true_count':
                true_count,

            'false_count':
                false_count,

            'missing_count':
                missing_count
        }
    )


meal_summary = (
    pd.DataFrame(meal_summary)
    .sort_values(
        'coverage_percent',
        ascending=False
    )
    .reset_index(drop=True)
)

meal_summary

In [ ]:
## Organize features created

attribute_feature_groups = {

    'boolean_features':
        boolean_feature_columns,

    'price_features': [
        'price_range',
        'price_category'
    ],

    'categorical_features': [
        'alcohol',
        'wifi',
        'noise_level',
        'attire'
    ],

    'parking_features':
        parking_feature_columns,

    'ambience_features':
        ambience_feature_columns,

    'meal_features':
        meal_feature_columns
}



In [ ]:
for group, features in attribute_feature_groups.items():

    print(
        f"{group}: {len(features)} features"
    )

In [ ]:
all_engineered_attributes = [
    feature
    for features
    in attribute_feature_groups.values()
    for feature
    in features
]

print(
    "\nTotal engineered attribute columns:",
    len(all_engineered_attributes)
)

In [ ]:
## Create Master Table

final_attribute_audit = []

for column in all_engineered_attributes:

    available_count = (
        philly_business[column]
        .notna()
        .sum()
    )

    missing_count = (
        philly_business[column]
        .isna()
        .sum()
    )

    final_attribute_audit.append(
        {
            'feature':
                column,

            'available_count':
                available_count,

            'coverage_percent':
                round(
                    available_count
                    / len(philly_business)
                    * 100,
                    2
                ),

            'missing_count':
                missing_count,

            'missing_percent':
                round(
                    missing_count
                    / len(philly_business)
                    * 100,
                    2
                )
        }
    )


final_attribute_audit = (
    pd.DataFrame(
        final_attribute_audit
    )
    .sort_values(
        'coverage_percent',
        ascending=False
    )
    .reset_index(drop=True)
)

final_attribute_audit

In [ ]:
## Check Data Types
philly_business[
    all_engineered_attributes
].dtypes

In [ ]:
## remove temp columns
helper_columns = [
    'parking_dict',
    'ambience_dict',
    'meal_dict'
]

philly_business.drop(
    columns=helper_columns,
    inplace=True,
    errors='ignore'
)

Restaurant attributes were extracted from Yelp's semi-structured attributes field and transformed into analysis-ready features representing service offerings, pricing and positioning, amenities, parking, ambience, and meal occasions. Inconsistent representations of Boolean and categorical values were standardized, while nested attributes such as BusinessParking, Ambience, and GoodForMeal were safely parsed and decomposed into individual variables. Missing information was explicitly preserved rather than interpreted as a negative response, maintaining the distinction between an attribute known to be absent and one for which Yelp provided no information. Feature-level coverage was subsequently audited to identify attributes suitable for descriptive analysis and later predictive modeling.

## Summary Statistics of the Data

In [ ]:
## Review what is already created / established
print("Philadelphia restaurant Review:")

print(
    f"Number of reviews: "
    f"{len(philly_reviews):,}"
)

print(
    f"Unique reviewers: "
    f"{philly_reviews['user_id'].nunique():,}"
)

print(
    f"Restaurants represented: "
    f"{philly_reviews['business_id'].nunique():,}"
)

print(
    f"Missing review text: "
    f"{philly_reviews['text'].isna().sum():,}"
)

print(
    f"Empty review text: "
    f"{philly_reviews['text'].fillna('').str.strip().eq('').sum():,}"
)

In [ ]:
## Review Length Stats
review_length_summary = (
    philly_reviews[
        [
            'review_word_count',
            'review_character_count'
        ]
    ]
    .describe()
    .round(2)
)

review_length_summary

In [ ]:
average_review_words = (
    philly_reviews[
        'review_word_count'
    ].mean()
)

median_review_words = (
    philly_reviews[
        'review_word_count'
    ].median()
)

average_review_characters = (
    philly_reviews[
        'review_character_count'
    ].mean()
)

print(
    f"Average review length: "
    f"{average_review_words:.2f} words"
)

print(
    f"Median review length: "
    f"{median_review_words:.0f} words"
)

print(
    f"Average review length: "
    f"{average_review_characters:.2f} characters"
)

In [ ]:
## Text normalization
philly_reviews[
    'text_clean'
] = (
    philly_reviews[
        'text'
    ]
    .fillna('')
    .str.lower()
)

#### Tokenization

In [ ]:
!python -m spacy download en_core_web_lg

In [ ]:
import spacy

nlp = spacy.load("en_core_web_lg")

In [ ]:
## Text Processing
tokens = []
lemmas = []
pos_tags = []

for doc in nlp.pipe(
    philly_reviews['text_clean'],
    batch_size=50
):

    # Store individual tokens
    tokens.append([
        token.text
        for token in doc
    ])

    # Store lemmatized versions
    lemmas.append([
        token.lemma_
        for token in doc
    ])

    # Store token and part-of-speech tag
    pos_tags.append([
        (token.text, token.pos_)
        for token in doc
    ])

In [ ]:
philly_reviews['tokens'] = tokens
philly_reviews['lemmas'] = lemmas
philly_reviews['pos_tags'] = pos_tags

In [ ]:
display(
    philly_reviews[
        [
            'text',
            'tokens',
            'lemmas',
            'pos_tags'
        ]
    ].head()
)

In [ ]:
word_tokens = []

for doc in nlp.pipe(
    philly_reviews['text_clean'],
    batch_size=50
):

    word_tokens.append([
        token.text.lower()
        for token in doc
        if token.is_alpha
    ])

philly_reviews['word_tokens'] = word_tokens

In [ ]:
all_words = []

for review in philly_reviews['word_tokens']:
    for word in review:
        all_words.append(word)

In [ ]:
total_tokens = len(all_words)

print(
    "Total number of tokens:",
    f"{total_tokens:,}"
)

In [ ]:
unique_words = set(all_words)

vocabulary_size = len(unique_words)

print(
    "Number of unique words:",
    f"{vocabulary_size:,}"
)

In [ ]:
philly_reviews['token_count'] = (
    philly_reviews['word_tokens']
    .apply(len)
)

In [ ]:
print(
    "Average review length:",
    round(
        philly_reviews['token_count'].mean(),
        2
    ),
    "words"
)

print(
    "Median review length:",
    philly_reviews['token_count'].median(),
    "words"
)

In [ ]:
philly_reviews['token_count'].describe()

In [ ]:
## Create clean Lemma tokens

clean_tokens = []

for doc in nlp.pipe(
    philly_reviews['text_clean'],
    batch_size=50
):

    review_tokens = []

    for token in doc:

        # Keep alphabetic words
        # Remove stopwords
        # Store the lemma rather than original word
        if token.is_alpha and not token.is_stop:

            review_tokens.append(
                token.lemma_.lower()
            )

    clean_tokens.append(
        review_tokens
    )


philly_reviews[
    'clean_tokens'
] = clean_tokens

In [ ]:
## Word Frequency
from collections import Counter

all_clean_words = []

for review in philly_reviews[
    'clean_tokens'
]:

    for word in review:

        all_clean_words.append(
            word
        )

In [ ]:
word_counts = Counter(
    all_clean_words
)

top_words = (
    word_counts
    .most_common(30)
)

top_words

In [ ]:
## Convert to dataframe
top_words_df = pd.DataFrame(
    top_words,
    columns=[
        'word',
        'frequency'
    ]
)

top_words_df

In [ ]:
## Extract Nouns
philly_reviews['nouns'] = (
    philly_reviews['pos_tags']
    .apply(
        lambda x: [
            word.lower()
            for word, pos in x
            if pos == 'NOUN'
        ]
    )
)

In [ ]:
## Combined
all_nouns = []

for review in philly_reviews['nouns']:

    for word in review:

        all_nouns.append(word)

In [ ]:
## Count
noun_counts = Counter(all_nouns)

top_nouns = noun_counts.most_common(20)

top_nouns_df = pd.DataFrame(
    top_nouns,
    columns=[
        'noun',
        'frequency'
    ]
)

top_nouns_df

In [ ]:
## Now Adjectives
philly_reviews['adjectives'] = (
    philly_reviews['pos_tags']
    .apply(
        lambda x: [
            word.lower()
            for word, pos in x
            if pos == 'ADJ'
        ]
    )
)

In [ ]:
all_adjectives = []

for review in philly_reviews['adjectives']:

    for word in review:

        all_adjectives.append(word)

In [ ]:
adjective_counts = Counter(
    all_adjectives
)

top_adjectives = (
    adjective_counts
    .most_common(20)
)

top_adjectives_df = pd.DataFrame(
    top_adjectives,
    columns=[
        'adjective',
        'frequency'
    ]
)

top_adjectives_df

In [ ]:
## Create clean text for bigram
philly_reviews['clean_text'] = (
    philly_reviews['clean_tokens']
    .apply(
        lambda x: ' '.join(x)
    )
)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

bigram_vectorizer = CountVectorizer(
    ngram_range=(2, 2)
)

bigram_matrix = (
    bigram_vectorizer
    .fit_transform(
        philly_reviews['clean_text']
    )
)

In [ ]:
## get bigram names
bigram_names = (
    bigram_vectorizer
    .get_feature_names_out()
)

In [ ]:
## Count
bigram_frequencies = (
    bigram_matrix
    .sum(axis=0)
    .A1
)

In [ ]:
## Convert to dataframe
bigram_df = pd.DataFrame(
    {
        'bigram':
            bigram_names,

        'frequency':
            bigram_frequencies
    }
)

In [ ]:
top_bigrams_df = (
    bigram_df
    .sort_values(
        'frequency',
        ascending=False
    )
    .head(20)
    .reset_index(drop=True)
)

top_bigrams_df

In [ ]:
## Review Start Rating Distibution
star_distribution = (
    philly_reviews['review_stars']
    .value_counts()
    .sort_index()
)

star_distribution

In [ ]:
star_percent = (
    philly_reviews['review_stars']
    .value_counts(normalize=True)
    .sort_index()
    * 100
).round(2)

star_percent

In [ ]:
average_review_stars = (
    philly_reviews['review_stars']
    .mean()
)

print(
    "Average review rating:",
    round(average_review_stars, 2)
)

In [ ]:
philly_reviews[
    'rating_class'
].value_counts(dropna=False)

In [ ]:
## Review length by rating class
review_length_by_rating = (
    philly_reviews
    .groupby('rating_class')['token_count']
    .agg([
        'count',
        'mean',
        'median',
        'std'
    ])
    .round(2)
)

review_length_by_rating

In [ ]:
## Seperte positive and negative reviews
positive_reviews = philly_reviews[
    philly_reviews['review_stars'] >= 4
]

negative_reviews = philly_reviews[
    philly_reviews['review_stars'] <= 2
]

In [ ]:
print(
    "Positive reviews:",
    f"{len(positive_reviews):,}"
)

print(
    "Negative reviews:",
    f"{len(negative_reviews):,}"
)

In [ ]:
# Top positive words
positive_words = []

for review in positive_reviews['clean_tokens']:

    for word in review:

        positive_words.append(word)


positive_word_counts = Counter(
    positive_words
)

top_positive_words = pd.DataFrame(
    positive_word_counts.most_common(20),
    columns=[
        'word',
        'frequency'
    ]
)

top_positive_words

In [ ]:
## Top negative words
negative_words = []

for review in negative_reviews['clean_tokens']:

    for word in review:

        negative_words.append(word)


negative_word_counts = Counter(
    negative_words
)

top_negative_words = pd.DataFrame(
    negative_word_counts.most_common(20),
    columns=[
        'word',
        'frequency'
    ]
)

top_negative_words

In [ ]:
## Normalize word frequencies
positive_total_words = len(positive_words)
negative_total_words = len(negative_words)

print(
    "Positive cleaned tokens:",
    f"{positive_total_words:,}"
)

print(
    "Negative cleaned tokens:",
    f"{negative_total_words:,}"
)

In [ ]:
positive_frequency = pd.DataFrame(
    positive_word_counts.items(),
    columns=['word', 'count']
)

positive_frequency['per_1000_words'] = (
    positive_frequency['count']
    / positive_total_words
    * 1000
)


negative_frequency = pd.DataFrame(
    negative_word_counts.items(),
    columns=['word', 'count']
)

negative_frequency['per_1000_words'] = (
    negative_frequency['count']
    / negative_total_words
    * 1000
)

In [ ]:
top_positive_normalized = (
    positive_frequency
    .sort_values(
        'per_1000_words',
        ascending=False
    )
    .head(20)
    .reset_index(drop=True)
)

top_positive_normalized

In [ ]:
top_negative_normalized = (
    negative_frequency
    .sort_values(
        'per_1000_words',
        ascending=False
    )
    .head(20)
    .reset_index(drop=True)
)

top_negative_normalized

In [ ]:
## Postive Bigram
# ============================================================
# STEP 3.5.11A — POSITIVE REVIEW BIGRAMS
# ============================================================

positive_bigram_vectorizer = CountVectorizer(
    ngram_range=(2, 2)
)

positive_bigram_matrix = (
    positive_bigram_vectorizer
    .fit_transform(
        positive_reviews['clean_text']
    )
)

positive_bigram_names = (
    positive_bigram_vectorizer
    .get_feature_names_out()
)

positive_bigram_frequencies = (
    positive_bigram_matrix
    .sum(axis=0)
    .A1
)

positive_bigrams = pd.DataFrame(
    {
        'bigram': positive_bigram_names,
        'frequency': positive_bigram_frequencies
    }
)

top_positive_bigrams = (
    positive_bigrams
    .sort_values(
        'frequency',
        ascending=False
    )
    .head(20)
    .reset_index(drop=True)
)

top_positive_bigrams

In [ ]:
## Negative Bigram
negative_bigram_vectorizer = CountVectorizer(
    ngram_range=(2, 2)
)

negative_bigram_matrix = (
    negative_bigram_vectorizer
    .fit_transform(
        negative_reviews['clean_text']
    )
)

negative_bigram_names = (
    negative_bigram_vectorizer
    .get_feature_names_out()
)

negative_bigram_frequencies = (
    negative_bigram_matrix
    .sum(axis=0)
    .A1
)

negative_bigrams = pd.DataFrame(
    {
        'bigram': negative_bigram_names,
        'frequency': negative_bigram_frequencies
    }
)

top_negative_bigrams = (
    negative_bigrams
    .sort_values(
        'frequency',
        ascending=False
    )
    .head(20)
    .reset_index(drop=True)
)

top_negative_bigrams

In [ ]:
## Final Summary Statistics

summary_statistics = pd.DataFrame({

    'Metric': [
        'Number of Reviews',
        'Total Word Tokens',
        'Vocabulary Size',
        'Average Review Length',
        'Median Review Length',
        'Unique Customers',
        'Restaurants Covered',
        'Average Review Rating',
        'Positive Reviews',
        'Neutral Reviews',
        'Negative Reviews'
    ],

    'Value': [
        len(philly_reviews),
        total_tokens,
        vocabulary_size,
        round(philly_reviews['token_count'].mean(), 2),
        philly_reviews['token_count'].median(),
        philly_reviews['user_id'].nunique(),
        philly_reviews['business_id'].nunique(),
        round(philly_reviews['review_stars'].mean(), 2),
        (philly_reviews['rating_class'] == 'Positive').sum(),
        (philly_reviews['rating_class'] == 'Neutral').sum(),
        (philly_reviews['rating_class'] == 'Negative').sum()
    ]
})

summary_statistics

In [ ]:
## Engagement Stats

vote_summary = (
    philly_reviews[
        [
            'useful',
            'funny',
            'cool'
        ]
    ]
    .agg([
        'mean',
        'median',
        'max'
    ])
    .round(2)
)

vote_summary

In [ ]:
philly_reviews['total_votes'] = (
    philly_reviews['useful']
    + philly_reviews['funny']
    + philly_reviews['cool']
)

print(
    "Average total votes per review:",
    round(
        philly_reviews['total_votes'].mean(),
        2
    )
)

In [ ]:
## Star Distribution Plot
import matplotlib.pyplot as plt

star_counts = (
    philly_reviews['review_stars']
    .value_counts()
    .sort_index()
)

plt.figure(
    figsize=(8, 5)
)

plt.bar(
    star_counts.index,
    star_counts.values
)

plt.title(
    'Distribution of Philadelphia Restaurant Review Ratings'
)

plt.xlabel(
    'Review Stars'
)

plt.ylabel(
    'Number of Reviews'
)

plt.xticks(
    [1, 2, 3, 4, 5]
)

plt.show()

In [ ]:
## review length plot

plt.figure(
    figsize=(9, 5)
)

plt.hist(
    philly_reviews['token_count'],
    bins=40,
    edgecolor='black'
)

plt.axvline(
    philly_reviews['token_count'].mean(),
    linestyle='--',
    label='Mean'
)

plt.axvline(
    philly_reviews['token_count'].median(),
    linestyle='--',
    label='Median'
)

plt.title(
    'Distribution of Philadelphia Restaurant Review Length'
)

plt.xlabel(
    'Review Length (Words)'
)

plt.ylabel(
    'Number of Reviews'
)

plt.legend()

plt.show()

In [ ]:
## Review volume over time plot

reviews_by_year = (
    philly_reviews
    .groupby('year')
    .size()
)

plt.figure(
    figsize=(10, 5)
)

plt.plot(
    reviews_by_year.index,
    reviews_by_year.values,
    marker='o'
)

plt.title(
    'Philadelphia Restaurant Review Volume by Year'
)

plt.xlabel(
    'Year'
)

plt.ylabel(
    'Number of Reviews'
)

plt.xticks(
    reviews_by_year.index,
    rotation=45
)

plt.show()

In [ ]:
## Top restaurant categories plot

top_categories = (
    category_frequency[
        category_frequency['category'] != 'Restaurants'
    ]
    .head(15)
)

plt.figure(
    figsize=(10, 6)
)

plt.barh(
    top_categories['category'],
    top_categories['restaurant_count']
)

plt.gca().invert_yaxis()

plt.title(
    'Most Common Philadelphia Restaurant Categories'
)

plt.xlabel(
    'Number of Restaurants'
)

plt.ylabel(
    'Category'
)

plt.show()

The summary statistics show that the Philadelphia restaurant dataset provides a large and information-rich sample for text analytics, consisting of 12,498 reviews across 484 restaurants and 9,963 unique customers. The review corpus contains approximately 1.38 million word tokens and 24,317 unique words, providing substantial textual variation for subsequent NLP analysis. Reviews average 110.38 words, with a median of 81 words, indicating a right-skewed distribution in which most reviews are moderate in length but some customers provide substantially more detailed feedback. Review engagement is relatively limited overall, with averages of 0.85 useful, 0.29 funny, and 0.40 cool votes per review and medians of zero across all three measures.

Customer ratings are generally favorable, with an overall average of 3.83 stars. Approximately 69.57% of reviews are positive (4–5 stars), compared with 16.74% negative (1–2 stars) and 13.69% neutral (3 stars), indicating a notable positive-rating imbalance that should be considered in later analyses. Review length also varies by satisfaction: negative reviews average 126.56 words and neutral reviews 125.30 words, compared with only 103.55 words for positive reviews. This suggests that customers with negative or mixed experiences tend to provide more detailed feedback, making both review length and rating class potentially valuable features for the later text analytics and predictive modeling stages of the project.

## Data Evaluation

The selected Yelp data is well suited to the business objective because they connect what customers say in reviews with how they evaluate restaurants through star ratings, while the business, user, tip, and check-in datasets provide additional context about restaurant characteristics and customer engagement. This combination supports analysis of the factors associated with positive and negative restaurant experiences rather than relying on ratings alone. The sample size is also appropriate for the proposed analysis: the dataset contains enough restaurants, customers, and especially review text to support NLP techniques and later machine-learning models while remaining computationally manageable in Python. The sample also includes restaurants across numerous categories and dining formats, providing meaningful variation within the Philadelphia restaurant market.

There are, however, several data-quality limitations that must be considered when interpreting results. Yelp reviews represent self-selected customers, meaning reviewers may differ from the broader population of restaurant customers, and the rating distribution is weighted toward positive experiences. Restaurants also receive unequal numbers of reviews, which could cause highly reviewed establishments to exert greater influence on aggregate results. From a processing perspective, the original Yelp JSON data contain nested and semi-structured fields, inconsistent representations of categorical and Boolean attributes, multi-label restaurant categories, and varying levels of missing information. Review text introduces additional challenges such as punctuation, stopwords, different word forms, varying review lengths, and context-dependent language. Consequently, careful feature engineering, text preprocessing, treatment of missing values, and consideration of class imbalance will be necessary before applying NLP and machine-learning methods.

## Prelimiary Data Exploration

Preliminary text exploration was conducted using spaCy and scikit-learn to transform the unstructured review text into analyzable features. Reviews were normalized to lowercase and processed through tokenization, alphabetic-word filtering, stopword removal, and lemmatization. Part-of-speech tagging was then used to separately examine frequently discussed nouns and descriptive adjectives, while CountVectorizer was used to identify common bigrams. The overall results indicate that customers primarily discuss food quality, service, wait/time, menu items, staff interactions, and the overall dining experience. The most frequent nouns included food, place, service, time, chicken, and menu, while prominent adjectives included good, great, delicious, nice, friendly, amazing, and fresh. Bigrams added further context, with phrases such as great food, great service, staff friendly, and highly recommend appearing frequently.

To better understand differences between satisfied and dissatisfied customers, reviews were also separated into positive (4–5 stars) and negative (1–2 stars) groups and word frequencies were normalized to account for the unequal group sizes. Positive reviews were characterized by quality and advocacy-oriented language such as great, delicious, and love, with bigrams including highly recommend, great food, great service, and staff friendly. Negative reviews showed a stronger emphasis on service execution, ordering, and waiting, with terms such as wait, ask, table, bad, and minute and phrases including wait minute, minute later, place order, take order, and long time. Overall, the preliminary analysis suggests that positive experiences are strongly associated with food quality and favorable service interactions, while dissatisfaction may be more closely connected to operational issues such as delays and the ordering/service process. These findings provide initial themes that can be examined more rigorously through the sentiment, feature-extraction, and predictive-modeling stages of the project.

## Proposed Solution

The next stages of the project will use a combination of unstructured data analytics and machine-learning techniques to identify the factors associated with positive and negative restaurant experiences. Review text will be transformed into quantitative features using methods such as TF-IDF, allowing important words and phrases to be weighted based on their relevance rather than raw frequency alone. Sentiment analysis will be used to measure the emotional tone expressed within each review and compared with the customer's actual star rating. Additional text analysis, such as topic modeling or clustering, can be used to identify recurring customer-experience themes such as food quality, service, wait times, pricing, and atmosphere. These methods will extend the preliminary frequency and bigram analysis by providing a more systematic way to determine what customers discuss and how those themes relate to satisfaction.

For the machine-learning portion, supervised classification models can be developed to predict whether a review represents a positive or negative customer experience using the text-derived features. An interpretable baseline model such as Logistic Regression can be compared with more advanced classification approaches, with performance evaluated using measures such as precision, recall, F1-score, confusion matrices, and ROC-AUC rather than accuracy alone because the rating classes are imbalanced. Text features can also be combined with structured restaurant characteristics, such as price range, delivery, reservations, parking, and other available attributes, to investigate whether restaurant characteristics improve predictive performance. Ultimately, the proposed approach is intended not only to predict customer sentiment, but also to identify actionable drivers of satisfaction and dissatisfaction that restaurant managers could use to improve the customer experience.

In [ ]:
## Look at hours
print(
    "Restaurants:",
    len(philly_business)
)

print(
    "Restaurants with hours:",
    philly_business['hours'].notna().sum()
)

print(
    "Restaurants missing hours:",
    philly_business['hours'].isna().sum()
)

print(
    "Hours coverage:",
    round(
        philly_business['hours'].notna().mean()
        * 100,
        2
    ),
    "%"
)

philly_business[
    [
        'name',
        'hours'
    ]
].head(10)

In [ ]:
# Calculate Days Open per Week
def count_days_open(hours):

    if not isinstance(hours, dict):
        return pd.NA

    return len(hours)


philly_business[
    'days_open_per_week'
] = (
    philly_business['hours']
    .apply(count_days_open)
    .astype('Int64')
)

philly_business[
    'days_open_per_week'
].value_counts(
    dropna=False
).sort_index()

philly_business[
    'days_open_per_week'
].describe()

In [ ]:
## Calculate Weekend Avaiability
def check_weekend_hours(hours):

    if not isinstance(hours, dict):
        return pd.NA

    if (
        'Saturday' in hours
        or 'Sunday' in hours
    ):
        return True

    return False


philly_business[
    'open_weekends'
] = (
    philly_business['hours']
    .apply(check_weekend_hours)
    .astype('boolean')
)

philly_business[
    'open_weekends'
].value_counts(
    dropna=False
)

In [ ]:
## Calculate weekly Hours
def calculate_daily_hours(hours_string):

    start_time, end_time = hours_string.split('-')

    start_hour, start_minute = map(
        int,
        start_time.split(':')
    )

    end_hour, end_minute = map(
        int,
        end_time.split(':')
    )

    start = start_hour + (start_minute / 60)
    end = end_hour + (end_minute / 60)

    # Account for restaurants closing after midnight
    if end < start:
        end += 24

    return end - start

In [ ]:
## Calculate hours per restaurant
def calculate_weekly_hours(hours):

    if not isinstance(hours, dict):
        return np.nan

    total_hours = 0

    for hours_string in hours.values():

        total_hours += calculate_daily_hours(
            hours_string
        )

    return total_hours


philly_business[
    'weekly_open_hours'
] = (
    philly_business['hours']
    .apply(calculate_weekly_hours)
)

In [ ]:
philly_business[
    [
        'name',
        'days_open_per_week',
        'weekly_open_hours',
        'hours'
    ]
].sort_values(
    'weekly_open_hours'
).head(10)

In [ ]:
philly_business[
    [
        'name',
        'days_open_per_week',
        'weekly_open_hours',
        'hours'
    ]
].sort_values(
    'weekly_open_hours',
    ascending=False
).head(10)

In [ ]:
## Calculate daily operating hours
def calculate_daily_hours(hours_string):

    start_time, end_time = hours_string.split('-')

    start_hour, start_minute = map(
        int,
        start_time.split(':')
    )

    end_hour, end_minute = map(
        int,
        end_time.split(':')
    )

    start = start_hour + (start_minute / 60)
    end = end_hour + (end_minute / 60)

    ## Account for 24-hour restaurants
    if start == 0 and end == 0:
        return 24

    ## Account for restaurants closing after midnight
    if end < start:
        end += 24

    return end - start

In [ ]:
## Calculate weekly operating hours
philly_business[
    'weekly_open_hours'
] = (
    philly_business['hours']
    .apply(calculate_weekly_hours)
)

In [ ]:
## Review weekly operating hours
philly_business[
    'weekly_open_hours'
].describe()

In [ ]:
## Restaurants with the most weekly operating hours
philly_business[
    [
        'name',
        'days_open_per_week',
        'weekly_open_hours'
    ]
].sort_values(
    'weekly_open_hours',
    ascending=False
).head(15)

## Create Modeling Dataset

In [ ]:
## Create modeling dataset
modeling_data = philly_reviews[
    [
        'review_id',
        'user_id',
        'business_id',
        'review_stars',
        'text',
        'clean_text',
        'token_count',
        'rating_class',
        'year'
    ]
].copy()

modeling_data.head()

In [ ]:
## Review modeling dataset
print(
    "Rows:",
    modeling_data.shape[0]
)

print(
    "Columns:",
    modeling_data.shape[1]
)

print(
    "\nMissing values:"
)

print(
    modeling_data
    .isna()
    .sum()
)

In [ ]:
## Review rating classes
modeling_data[
    'rating_class'
].value_counts()

In [ ]:
## Review business feature names
philly_business.columns.tolist()

In [ ]:
## Select restaurant features for review-level analysis
business_features = philly_business[
    [
        'business_id',
        'name',
        'business_stars',
        'business_review_count',
        'is_open',
        'price_range',
        'cuisine_groups',
        'weekly_open_hours'
    ]
].copy()

In [ ]:
## Merge restaurant features with review data
modeling_data = modeling_data.merge(
    business_features,
    on='business_id',
    how='left'
)

modeling_data.head()

In [ ]:
## Review merged modeling dataset
print(
    "Rows:",
    modeling_data.shape[0]
)

print(
    "Columns:",
    modeling_data.shape[1]
)

print(
    "\nUnique restaurants:",
    modeling_data['business_id'].nunique()
)

print(
    "\nMissing values:"
)

print(
    modeling_data
    .isna()
    .sum()
)

## NLP Feature Extraction and Sentiment Analysis

In [ ]:
## Create positive and negative review dataset
classification_data = modeling_data[
    modeling_data['rating_class'] != 'Neutral'
].copy()

classification_data[
    'rating_class'
].value_counts()

In [ ]:
## Create binary target variable
classification_data[
    'positive_review'
] = (
    classification_data[
        'rating_class'
    ] == 'Positive'
).astype(int)

classification_data[
    'positive_review'
].value_counts()

In [ ]:
## Train Test Split
from sklearn.model_selection import train_test_split

## Define review text and target
X = classification_data[
    'clean_text'
]

y = classification_data[
    'positive_review'
]

## Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(
    "Training reviews:",
    len(X_train)
)

print(
    "Testing reviews:",
    len(X_test)
)

print(
    "\nTraining class distribution:"
)

print(
    y_train.value_counts()
)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

## Create TF-IDF vectorizer
tfidf = TfidfVectorizer(
    max_features=5000,
    min_df=5,
    max_df=0.90,
    ngram_range=(1, 2)
)

## Fit TF-IDF to training reviews
X_train_tfidf = tfidf.fit_transform(
    X_train
)

## Transform testing reviews
X_test_tfidf = tfidf.transform(
    X_test
)

In [ ]:
## Review TF-IDF matrix dimensions
print(
    "Training TF-IDF shape:",
    X_train_tfidf.shape
)

print(
    "Testing TF-IDF shape:",
    X_test_tfidf.shape
)

print(
    "Number of TF-IDF features:",
    len(
        tfidf.get_feature_names_out()
    )
)

In [ ]:
## Review sample TF-IDF features
tfidf_features = tfidf.get_feature_names_out()

print(
    tfidf_features[:50]
)

In [ ]:
!pip install vaderSentiment

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

## Create sentiment analyzer
sentiment_analyzer = SentimentIntensityAnalyzer()

In [ ]:
## Calculate sentiment scores
modeling_data[
    'sentiment_score'
] = modeling_data[
    'text'
].apply(
    lambda review:
    sentiment_analyzer
    .polarity_scores(review)[
        'compound'
    ]
)

In [ ]:
## Review sentiment score distribution
modeling_data[
    'sentiment_score'
].describe()

In [ ]:
## Compare sentiment with star ratings
sentiment_by_rating = (
    modeling_data
    .groupby('review_stars')[
        'sentiment_score'
    ]
    .agg(
        [
            'count',
            'mean',
            'median'
        ]
    )
)

sentiment_by_rating

Sentiment increases as reviews are more positive as predicted

In [ ]:
## Calculate correlation between sentiment and star rating
sentiment_rating_correlation = (
    modeling_data[
        [
            'sentiment_score',
            'review_stars'
        ]
    ]
    .corr()
)

sentiment_rating_correlation

VADER sentiment analysis showed a clear positive relationship between review language and Yelp star ratings. Mean sentiment increased from -0.114 for one-star reviews to 0.892 for five-star reviews, and sentiment scores had a positive correlation of 0.578 with star ratings. Although the relationship was substantial, sentiment did not perfectly correspond with ratings, particularly among two- and three-star reviews. This suggests that customers often use mixed positive and negative language even when their overall rating reflects dissatisfaction or neutrality. Therefore, Yelp ratings were retained as the target for supervised classification, while VADER sentiment was used as a complementary measure of customer experience.

In [ ]:
## Visualize sentiment by star rating
sentiment_plot_data = [
    modeling_data[
        modeling_data['review_stars'] == rating
    ]['sentiment_score']
    for rating in range(1, 6)
]

plt.figure(
    figsize=(9, 6)
)

plt.boxplot(
    sentiment_plot_data,
    labels=[
        '1 Star',
        '2 Stars',
        '3 Stars',
        '4 Stars',
        '5 Stars'
    ]
)

plt.axhline(
    y=0,
    linestyle='--'
)

plt.title(
    'Review Sentiment by Yelp Star Rating'
)

plt.xlabel(
    'Yelp Star Rating'
)

plt.ylabel(
    'VADER Compound Sentiment Score'
)

plt.show()

## Satisfaction Prediction

### Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

## Create Logistic Regression model
logistic_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42
)

## Train Logistic Regression model
logistic_model.fit(
    X_train_tfidf,
    y_train
)

In [ ]:
## Generate Logistic Regression predictions
logistic_predictions = logistic_model.predict(
    X_test_tfidf
)

logistic_probabilities = logistic_model.predict_proba(
    X_test_tfidf
)[:, 1]

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

## Calculate Logistic Regression performance
logistic_accuracy = accuracy_score(
    y_test,
    logistic_predictions
)

logistic_precision = precision_score(
    y_test,
    logistic_predictions
)

logistic_recall = recall_score(
    y_test,
    logistic_predictions
)

logistic_f1 = f1_score(
    y_test,
    logistic_predictions
)

logistic_roc_auc = roc_auc_score(
    y_test,
    logistic_probabilities
)

print(
    "Accuracy:",
    round(logistic_accuracy, 4)
)

print(
    "Precision:",
    round(logistic_precision, 4)
)

print(
    "Recall:",
    round(logistic_recall, 4)
)

print(
    "F1 Score:",
    round(logistic_f1, 4)
)

print(
    "ROC-AUC:",
    round(logistic_roc_auc, 4)
)

In [ ]:
## Review classification performance
print(
    classification_report(
        y_test,
        logistic_predictions,
        target_names=[
            'Negative',
            'Positive'
        ]
    )
)

In [ ]:
## Create confusion matrix
logistic_confusion_matrix = confusion_matrix(
    y_test,
    logistic_predictions
)

logistic_confusion_matrix

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

## Visualize Logistic Regression confusion matrix
confusion_display = ConfusionMatrixDisplay(
    confusion_matrix=logistic_confusion_matrix,
    display_labels=[
        'Negative',
        'Positive'
    ]
)

confusion_display.plot()

plt.title(
    'Logistic Regression Confusion Matrix'
)

plt.show()

In [ ]:
from sklearn.metrics import roc_curve

## Calculate ROC curve
false_positive_rate, true_positive_rate, thresholds = roc_curve(
    y_test,
    logistic_probabilities
)

## Visualize ROC curve
plt.figure(
    figsize=(8, 6)
)

plt.plot(
    false_positive_rate,
    true_positive_rate,
    label='Logistic Regression'
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle='--',
    label='Random Classification'
)

plt.xlabel(
    'False Positive Rate'
)

plt.ylabel(
    'True Positive Rate'
)

plt.title(
    'Logistic Regression ROC Curve'
)

plt.legend()

plt.show()

The TF-IDF Logistic Regression model demonstrated strong performance in classifying customer satisfaction, achieving 93.1% accuracy and a ROC-AUC of 0.975 on the test data. Positive reviews were identified with 97% precision and 94% recall, while negative reviews achieved 78% precision and 89% recall. The model correctly identified 372 of 419 negative reviews, indicating that the class-weighted approach was effective at detecting dissatisfied customers despite the imbalance toward positive reviews. Overall, the results demonstrate that review language contains substantial information for distinguishing satisfied from dissatisfied customers. The model's strong ROC-AUC further indicates excellent separation between positive and negative review language

In [ ]:
## Create TF-IDF coefficient table
coefficient_data = pd.DataFrame(
    {
        'feature': tfidf.get_feature_names_out(),
        'coefficient': logistic_model.coef_[0]
    }
)

## Identify strongest negative review features
top_negative_features = (
    coefficient_data
    .sort_values(
        'coefficient',
        ascending=True
    )
    .head(20)
)

top_negative_features

In [ ]:
## Identify strongest positive review features
top_positive_features = (
    coefficient_data
    .sort_values(
        'coefficient',
        ascending=False
    )
    .head(20)
)

top_positive_features

In [ ]:
## Select strongest review features
coefficient_plot_data = pd.concat(
    [
        top_negative_features.head(10),
        top_positive_features.head(10)
    ]
).sort_values(
    'coefficient'
)

## Visualize strongest review features
plt.figure(
    figsize=(10, 8)
)

plt.barh(
    coefficient_plot_data['feature'],
    coefficient_plot_data['coefficient']
)

plt.axvline(
    x=0,
    linestyle='--'
)

plt.title(
    'Strongest Predictors of Customer Satisfaction'
)

plt.xlabel(
    'Logistic Regression Coefficient'
)

plt.ylabel(
    'TF-IDF Feature'
)

plt.show()

Logistic Regression coefficients were examined to identify the TF-IDF features most strongly associated with customer satisfaction and dissatisfaction. Positive reviews were strongly associated with terms such as delicious, great, amazing, love, excellent, and friendly, suggesting that food quality and positive service experiences are important components of customer satisfaction. Negative reviews were associated with terms including bad, bland, terrible, disappointing, dry, overpriced, cold, and rude. These results suggest that dissatisfaction is associated not only with overall negative evaluations, but also with specific issues involving food quality, perceived value, and service execution. These findings reinforce the earlier exploratory text analysis, which found that dissatisfied customers used more language related to ordering, waiting, and service interactions.

In [ ]:
from sklearn.neural_network import MLPClassifier

## Create Neural Network model
neural_network_model = MLPClassifier(
    hidden_layer_sizes=(100,),
    activation='relu',
    solver='adam',
    max_iter=100,
    early_stopping=True,
    random_state=42
)

## Train Neural Network model
neural_network_model.fit(
    X_train_tfidf,
    y_train
)

In [ ]:
## Generate Neural Network predictions
neural_network_predictions = neural_network_model.predict(
    X_test_tfidf
)

neural_network_probabilities = neural_network_model.predict_proba(
    X_test_tfidf
)[:, 1]

In [ ]:
## Calculate Neural Network performance
neural_network_accuracy = accuracy_score(
    y_test,
    neural_network_predictions
)

neural_network_precision = precision_score(
    y_test,
    neural_network_predictions
)

neural_network_recall = recall_score(
    y_test,
    neural_network_predictions
)

neural_network_f1 = f1_score(
    y_test,
    neural_network_predictions
)

neural_network_roc_auc = roc_auc_score(
    y_test,
    neural_network_probabilities
)

print(
    "Accuracy:",
    round(neural_network_accuracy, 4)
)

print(
    "Precision:",
    round(neural_network_precision, 4)
)

print(
    "Recall:",
    round(neural_network_recall, 4)
)

print(
    "F1 Score:",
    round(neural_network_f1, 4)
)

print(
    "ROC-AUC:",
    round(neural_network_roc_auc, 4)
)

In [ ]:
## Review Neural Network performance by rating class
print(
    classification_report(
        y_test,
        neural_network_predictions,
        target_names=[
            'Negative',
            'Positive'
        ]
    )
)

In [ ]:
## Create Neural Network confusion matrix
neural_network_confusion_matrix = confusion_matrix(
    y_test,
    neural_network_predictions
)

neural_network_confusion_matrix

In [ ]:
## Compare classification models
model_comparison = pd.DataFrame(
    {
        'Model': [
            'Logistic Regression',
            'Neural Network'
        ],
        'Accuracy': [
            logistic_accuracy,
            neural_network_accuracy
        ],
        'Precision': [
            logistic_precision,
            neural_network_precision
        ],
        'Recall': [
            logistic_recall,
            neural_network_recall
        ],
        'F1 Score': [
            logistic_f1,
            neural_network_f1
        ],
        'ROC-AUC': [
            logistic_roc_auc,
            neural_network_roc_auc
        ]
    }
)

model_comparison.round(4)

In [ ]:
## Calculate negative review recall
logistic_negative_recall = recall_score(
    y_test,
    logistic_predictions,
    pos_label=0
)

neural_network_negative_recall = recall_score(
    y_test,
    neural_network_predictions,
    pos_label=0
)

model_comparison[
    'Negative Recall'
] = [
    logistic_negative_recall,
    neural_network_negative_recall
]

model_comparison.round(4)

Both models demonstrated strong ability to classify customer satisfaction from review text. Logistic Regression achieved 93.1% accuracy and a ROC-AUC of 0.975, while the Neural Network achieved slightly higher accuracy of 93.4% with the same ROC-AUC of 0.975. Although the Neural Network produced marginally higher overall accuracy and positive-review recall, Logistic Regression identified 88.8% of negative reviews compared with 74.0% for the Neural Network. This distinction is important for customer-experience applications because failing to identify dissatisfied customers may prevent restaurants from recognizing recurring service or product-quality issues. Logistic Regression also provides directly interpretable coefficients that identify language associated with satisfaction and dissatisfaction. The results therefore show that increased model complexity did not produce a substantial overall performance improvement, while the two models exhibited different classification tradeoffs.

In [ ]:
## Calculate Neural Network ROC curve
nn_false_positive_rate, nn_true_positive_rate, nn_thresholds = roc_curve(
    y_test,
    neural_network_probabilities
)

## Compare model ROC curves
plt.figure(
    figsize=(8, 6)
)

plt.plot(
    false_positive_rate,
    true_positive_rate,
    label='Logistic Regression'
)

plt.plot(
    nn_false_positive_rate,
    nn_true_positive_rate,
    label='Neural Network'
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle='--',
    label='Random Classification'
)

plt.xlabel(
    'False Positive Rate'
)

plt.ylabel(
    'True Positive Rate'
)

plt.title(
    'ROC Curve Comparison'
)

plt.legend()

plt.show()

## Restaurant Segmentation and Clustering

In [ ]:
## Aggregate review features by restaurant
restaurant_review_features = (
    modeling_data
    .groupby('business_id')
    .agg(
        observed_reviews=(
            'review_id',
            'count'
        ),
        average_review_rating=(
            'review_stars',
            'mean'
        ),
        average_sentiment=(
            'sentiment_score',
            'mean'
        ),
        average_review_length=(
            'token_count',
            'mean'
        )
    )
    .reset_index()
)

restaurant_review_features.head()

In [ ]:
## Calculate review class counts by restaurant
restaurant_rating_counts = (
    modeling_data
    .groupby(
        [
            'business_id',
            'rating_class'
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
    .reset_index()
)

restaurant_rating_counts.head()

In [ ]:
## Calculate rating class proportions
restaurant_rating_counts[
    'positive_review_proportion'
] = (
    restaurant_rating_counts['Positive']
    /
    restaurant_rating_counts[
        [
            'Positive',
            'Neutral',
            'Negative'
        ]
    ].sum(axis=1)
)

restaurant_rating_counts[
    'negative_review_proportion'
] = (
    restaurant_rating_counts['Negative']
    /
    restaurant_rating_counts[
        [
            'Positive',
            'Neutral',
            'Negative'
        ]
    ].sum(axis=1)
)

In [ ]:
## Select restaurant satisfaction features
restaurant_satisfaction_features = (
    restaurant_rating_counts[
        [
            'business_id',
            'positive_review_proportion',
            'negative_review_proportion'
        ]
    ]
    .copy()
)

In [ ]:
## Combine restaurant review features
restaurant_features = (
    restaurant_review_features
    .merge(
        restaurant_satisfaction_features,
        on='business_id',
        how='left'
    )
)

restaurant_features.head()

In [ ]:
## Review restaurant-level dataset
print(
    "Restaurants:",
    restaurant_features.shape[0]
)

print(
    "Columns:",
    restaurant_features.shape[1]
)

print(
    "\nMissing values:"
)

print(
    restaurant_features
    .isna()
    .sum()
)

restaurant_features.describe()

In [ ]:
## Calculate check-ins per restaurant
restaurant_checkin_features = (
    philly_checkins
    .groupby('business_id')
    .size()
    .reset_index(
        name='observed_checkins'
    )
)

restaurant_checkin_features.head()

In [ ]:
## Add check-in activity
restaurant_features = (
    restaurant_features
    .merge(
        restaurant_checkin_features,
        on='business_id',
        how='left'
    )
)

In [ ]:
## Replace missing check-in counts with zero
restaurant_features[
    'observed_checkins'
] = (
    restaurant_features[
        'observed_checkins'
    ]
    .fillna(0)
)

In [ ]:
## Review restaurant check-in activity
restaurant_features[
    'observed_checkins'
].describe()

In [ ]:
## Count restaurants without observed check-ins
print(
    "Restaurants with no observed check-ins:",
    (
        restaurant_features[
            'observed_checkins'
        ] == 0
    ).sum()
)

In [ ]:
## Select restaurant characteristics
restaurant_business_features = philly_business[
    [
        'business_id',
        'name',
        'business_stars',
        'business_review_count',
        'is_open',
        'price_range',
        'cuisine_groups',
        'days_open_per_week',
        'open_weekends',
        'weekly_open_hours',
        'offers_delivery',
        'offers_takeout',
        'takes_reservations',
        'table_service',
        'good_for_groups',
        'outdoor_seating'
    ]
].copy()

In [ ]:
## Add restaurant characteristics
restaurant_features = (
    restaurant_features
    .merge(
        restaurant_business_features,
        on='business_id',
        how='left'
    )
)

In [ ]:
## Review restaurant clustering dataset
print(
    "Restaurants:",
    restaurant_features.shape[0]
)

print(
    "Columns:",
    restaurant_features.shape[1]
)

print(
    "\nMissing values:"
)

print(
    restaurant_features
    .isna()
    .sum()
)

In [ ]:
## Create transformed restaurant activity features
restaurant_features[
    'log_observed_reviews'
] = np.log1p(
    restaurant_features[
        'observed_reviews'
    ]
)

restaurant_features[
    'log_observed_checkins'
] = np.log1p(
    restaurant_features[
        'observed_checkins'
    ]
)## Create transformed restaurant activity features
restaurant_features[
    'log_observed_reviews'
] = np.log1p(
    restaurant_features[
        'observed_reviews'
    ]
)

restaurant_features[
    'log_observed_checkins'
] = np.log1p(
    restaurant_features[
        'observed_checkins'
    ]
)

In [ ]:
## Compare activity distributions
restaurant_features[
    [
        'observed_reviews',
        'log_observed_reviews',
        'observed_checkins',
        'log_observed_checkins'
    ]
].describe()

In [ ]:
## Review restaurant price distribution
restaurant_features[
    'price_range'
].value_counts(
    dropna=False
).sort_index()

In [ ]:
## Create clustering price feature
restaurant_features[
    'clustering_price_range'
] = restaurant_features[
    'price_range'
].fillna(
    restaurant_features[
        'price_range'
    ].median()
)

In [ ]:
## Select restaurant clustering features
clustering_features = [
    'average_review_rating',
    'average_sentiment',
    'negative_review_proportion',
    'average_review_length',
    'log_observed_reviews',
    'log_observed_checkins',
    'clustering_price_range'
]

restaurant_clustering_data = restaurant_features[
    clustering_features
].copy()

In [ ]:
## Review clustering dataset
print(
    "Restaurants:",
    restaurant_clustering_data.shape[0]
)

print(
    "Clustering features:",
    restaurant_clustering_data.shape[1]
)

print(
    "\nMissing values:"
)

print(
    restaurant_clustering_data
    .isna()
    .sum()
)

restaurant_clustering_data.describe()

In [ ]:
## Calculate clustering feature correlations
clustering_correlation = (
    restaurant_clustering_data
    .corr()
)

clustering_correlation.round(2)

In [ ]:
## Visualize clustering feature correlations
plt.figure(
    figsize=(9, 7)
)

plt.imshow(
    clustering_correlation,
    aspect='auto'
)

plt.colorbar(
    label='Correlation'
)

plt.xticks(
    range(
        len(clustering_correlation.columns)
    ),
    clustering_correlation.columns,
    rotation=90
)

plt.yticks(
    range(
        len(clustering_correlation.columns)
    ),
    clustering_correlation.columns
)

plt.title(
    'Restaurant Clustering Feature Correlations'
)

plt.show()

In [ ]:
## Select final restaurant clustering features
final_clustering_features = [
    'average_sentiment',
    'negative_review_proportion',
    'average_review_length',
    'log_observed_reviews',
    'log_observed_checkins',
    'clustering_price_range'
]

restaurant_clustering_data = restaurant_features[
    final_clustering_features
].copy()

restaurant_clustering_data.head()

In [ ]:
from sklearn.preprocessing import StandardScaler

## Standardize restaurant clustering features
clustering_scaler = StandardScaler()

restaurant_clustering_scaled = (
    clustering_scaler.fit_transform(
        restaurant_clustering_data
    )
)

restaurant_clustering_scaled.shape

In [ ]:
## Review standardized clustering features
scaled_clustering_data = pd.DataFrame(
    restaurant_clustering_scaled,
    columns=final_clustering_features
)

scaled_clustering_data.describe().round(2)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

## Evaluate possible numbers of clusters
cluster_results = []

for number_of_clusters in range(2, 11):

    kmeans_model = KMeans(
        n_clusters=number_of_clusters,
        random_state=42,
        n_init=10
    )

    cluster_labels = kmeans_model.fit_predict(
        restaurant_clustering_scaled
    )

    cluster_results.append(
        {
            'number_of_clusters': number_of_clusters,
            'inertia': kmeans_model.inertia_,
            'silhouette_score': silhouette_score(
                restaurant_clustering_scaled,
                cluster_labels
            )
        }
    )

cluster_evaluation = pd.DataFrame(
    cluster_results
)

cluster_evaluation

In [ ]:
## Visualize K-Means elbow method
plt.figure(
    figsize=(8, 6)
)

plt.plot(
    cluster_evaluation[
        'number_of_clusters'
    ],
    cluster_evaluation[
        'inertia'
    ],
    marker='o'
)

plt.xlabel(
    'Number of Clusters'
)

plt.ylabel(
    'Within-Cluster Sum of Squares'
)

plt.title(
    'K-Means Elbow Method'
)

plt.xticks(
    range(2, 11)
)

plt.show()

In [ ]:
## Visualize Silhouette Scores
plt.figure(
    figsize=(8, 6)
)

plt.plot(
    cluster_evaluation[
        'number_of_clusters'
    ],
    cluster_evaluation[
        'silhouette_score'
    ],
    marker='o'
)

plt.xlabel(
    'Number of Clusters'
)

plt.ylabel(
    'Silhouette Score'
)

plt.title(
    'K-Means Silhouette Analysis'
)

plt.xticks(
    range(2, 11)
)

plt.show()

In [ ]:
## Create final K-Means model
final_kmeans_model = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

## Assign restaurants to clusters
restaurant_features[
    'cluster'
] = final_kmeans_model.fit_predict(
    restaurant_clustering_scaled
)

restaurant_features[
    'cluster'
].value_counts().sort_index()

In [ ]:
## Profile restaurant clusters
cluster_profile = (
    restaurant_features
    .groupby('cluster')
    .agg(
        restaurants=(
            'business_id',
            'count'
        ),
        average_rating=(
            'average_review_rating',
            'mean'
        ),
        average_sentiment=(
            'average_sentiment',
            'mean'
        ),
        negative_review_proportion=(
            'negative_review_proportion',
            'mean'
        ),
        average_review_length=(
            'average_review_length',
            'mean'
        ),
        average_reviews=(
            'observed_reviews',
            'mean'
        ),
        average_checkins=(
            'observed_checkins',
            'mean'
        ),
        average_price=(
            'clustering_price_range',
            'mean'
        )
    )
    .round(2)
)

cluster_profile

In [ ]:
## Create standardized cluster center table
cluster_centers = pd.DataFrame(
    final_kmeans_model.cluster_centers_,
    columns=final_clustering_features
)

cluster_centers.index.name = 'cluster'

cluster_centers.round(2)

In [ ]:
## Profile restaurant service characteristics
cluster_service_profile = (
    restaurant_features
    .groupby('cluster')
    [
        [
            'offers_delivery',
            'offers_takeout',
            'takes_reservations',
            'table_service',
            'good_for_groups',
            'outdoor_seating'
        ]
    ]
    .mean()
    .round(2)
)

cluster_service_profile

In [ ]:
## Profile restaurant operating characteristics
cluster_operating_profile = (
    restaurant_features
    .groupby('cluster')
    .agg(
        average_days_open=(
            'days_open_per_week',
            'mean'
        ),
        average_weekly_hours=(
            'weekly_open_hours',
            'mean'
        )
    )
    .round(2)
)

cluster_operating_profile

In [ ]:
## Compare check-in availability with cluster assignment
checkin_cluster_table = pd.crosstab(
    restaurant_features['cluster'],
    restaurant_features['observed_checkins'] == 0
)

checkin_cluster_table.columns = [
    'Has Check-Ins',
    'No Check-Ins'
]

checkin_cluster_table

In [ ]:
## Review restaurants without observed check-ins
restaurant_features.loc[
    restaurant_features[
        'observed_checkins'
    ] == 0,
    [
        'name',
        'average_review_rating',
        'average_sentiment',
        'negative_review_proportion',
        'observed_reviews',
        'observed_checkins',
        'price_range',
        'is_open'
    ]
].sort_values(
    'observed_reviews',
    ascending=False
)

In [ ]:
## Select clustering features without check-ins
clustering_features_without_checkins = [
    'average_sentiment',
    'negative_review_proportion',
    'average_review_length',
    'log_observed_reviews',
    'clustering_price_range'
]

clustering_data_without_checkins = restaurant_features[
    clustering_features_without_checkins
].copy()

In [ ]:
## Standardize clustering features without check-ins
clustering_scaler_without_checkins = StandardScaler()

clustering_scaled_without_checkins = (
    clustering_scaler_without_checkins
    .fit_transform(
        clustering_data_without_checkins
    )
)

In [ ]:
## Evaluate cluster counts without check-ins
cluster_results_without_checkins = []

for number_of_clusters in range(2, 11):

    kmeans_model = KMeans(
        n_clusters=number_of_clusters,
        random_state=42,
        n_init=10
    )

    cluster_labels = kmeans_model.fit_predict(
        clustering_scaled_without_checkins
    )

    cluster_results_without_checkins.append(
        {
            'number_of_clusters':
                number_of_clusters,

            'inertia':
                kmeans_model.inertia_,

            'silhouette_score':
                silhouette_score(
                    clustering_scaled_without_checkins,
                    cluster_labels
                )
        }
    )

cluster_evaluation_without_checkins = pd.DataFrame(
    cluster_results_without_checkins
)

cluster_evaluation_without_checkins

In [ ]:
## Fit final K-Means clustering model
final_kmeans_model = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=10
)

restaurant_features[
    'final_cluster'
] = final_kmeans_model.fit_predict(
    clustering_scaled_without_checkins
)

restaurant_features[
    'final_cluster'
].value_counts().sort_index()

In [ ]:
## Profile final restaurant clusters
final_cluster_profile = (
    restaurant_features
    .groupby('final_cluster')
    .agg(
        restaurants=(
            'business_id',
            'count'
        ),
        average_rating=(
            'average_review_rating',
            'mean'
        ),
        average_sentiment=(
            'average_sentiment',
            'mean'
        ),
        negative_review_proportion=(
            'negative_review_proportion',
            'mean'
        ),
        average_review_length=(
            'average_review_length',
            'mean'
        ),
        average_reviews=(
            'observed_reviews',
            'mean'
        ),
        average_checkins=(
            'observed_checkins',
            'mean'
        ),
        average_price=(
            'clustering_price_range',
            'mean'
        ),
        average_weekly_hours=(
            'weekly_open_hours',
            'mean'
        )
    )
    .round(2)
)

final_cluster_profile

In [ ]:
## Create standardized final cluster centers
final_cluster_centers = pd.DataFrame(
    final_kmeans_model.cluster_centers_,
    columns=clustering_features_without_checkins
)

final_cluster_centers.index.name = 'final_cluster'

final_cluster_centers.round(2)

In [ ]:
## Profile service characteristics by final cluster
final_service_profile = (
    restaurant_features
    .groupby('final_cluster')
    [
        [
            'offers_delivery',
            'offers_takeout',
            'takes_reservations',
            'good_for_groups',
            'outdoor_seating'
        ]
    ]
    .mean()
    .round(2)
)

final_service_profile

In [ ]:
## Create review volume groups
restaurant_features[
    'review_volume_group'
] = pd.cut(
    restaurant_features[
        'observed_reviews'
    ],
    bins=[
        0,
        3,
        10,
        30,
        float('inf')
    ],
    labels=[
        '1-3 Reviews',
        '4-10 Reviews',
        '11-30 Reviews',
        '31+ Reviews'
    ]
)

## Compare review volume by cluster
review_volume_by_cluster = pd.crosstab(
    restaurant_features[
        'final_cluster'
    ],
    restaurant_features[
        'review_volume_group'
    ]
)

review_volume_by_cluster

In [ ]:
## Calculate review volume percentages by cluster
review_volume_percentages = pd.crosstab(
    restaurant_features[
        'final_cluster'
    ],
    restaurant_features[
        'review_volume_group'
    ],
    normalize='index'
) * 100

review_volume_percentages.round(1)

In [ ]:
## Compare ratings across final clusters
restaurant_features.groupby(
    'final_cluster'
)[
    'average_review_rating'
].agg(
    [
        'count',
        'mean',
        'median',
        'std'
    ]
).round(2)

In [ ]:
## Review restaurants in each cluster
representative_restaurants = restaurant_features[
    [
        'name',
        'final_cluster',
        'average_review_rating',
        'average_sentiment',
        'negative_review_proportion',
        'observed_reviews',
        'observed_checkins',
        'price_range',
        'cuisine_groups'
    ]
].sort_values(
    [
        'final_cluster',
        'observed_reviews'
    ],
    ascending=[
        True,
        False
    ]
)

representative_restaurants.groupby(
    'final_cluster'
).head(10)

In [ ]:
## Calculate total check-in events by restaurant
restaurant_checkin_counts = (
    philly_checkins
    .groupby('business_id')
    .size()
    .reset_index(
        name='total_checkins'
    )
)

restaurant_checkin_counts[
    'total_checkins'
].describe()

In [ ]:
## Add corrected check-in counts
restaurant_features = (
    restaurant_features
    .drop(
        columns='observed_checkins'
    )
    .merge(
        restaurant_checkin_counts,
        on='business_id',
        how='left'
    )
)

restaurant_features[
    'total_checkins'
] = (
    restaurant_features[
        'total_checkins'
    ]
    .fillna(0)
)

In [ ]:
## Review highest check-in activity restaurants
restaurant_features[
    [
        'name',
        'total_checkins'
    ]
].sort_values(
    'total_checkins',
    ascending=False
).head(10)

In [ ]:
## Review corrected check-in counts
restaurant_features[
    'total_checkins'
].describe()

In [ ]:
## Review restaurants with the most check-ins
restaurant_features[
    [
        'name',
        'total_checkins'
    ]
].sort_values(
    'total_checkins',
    ascending=False
).head(10)

In [ ]:
## Calculate total check-in events by restaurant
restaurant_checkin_counts = (
    philly_checkins
    .groupby('business_id')
    .size()
    .reset_index(
        name='total_checkins'
    )
)

In [ ]:
## Remove existing check-in count if already created
if 'total_checkins' in restaurant_features.columns:
    restaurant_features = (
        restaurant_features
        .drop(
            columns='total_checkins'
        )
    )

In [ ]:
## Add total check-in counts
restaurant_features = (
    restaurant_features
    .merge(
        restaurant_checkin_counts,
        on='business_id',
        how='left'
    )
)

restaurant_features[
    'total_checkins'
] = (
    restaurant_features[
        'total_checkins'
    ]
    .fillna(0)
)

Cluster 0 — Higher-Satisfaction, Higher-Engagement Restaurants contains 379 restaurants, or about 78.3% of the sample. Its standardized center shows higher sentiment (+0.38 SD), fewer negative reviews (−0.42 SD), and greater review activity (+0.20 SD). These restaurants also tended to have slightly higher price positioning (+0.10 SD). External profiling supported the segmentation: high-volume restaurants in this cluster generally showed ratings around 3.4–4.5 stars, strongly positive sentiment, and relatively small proportions of negative reviews. Operationally, 42% offered reservations and 39% outdoor seating.

Cluster 1 — Lower-Satisfaction, Lower-Engagement Restaurants contains 105 restaurants, or about 21.7% of the sample. Its defining characteristics were substantially lower sentiment (−1.38 SD), a much higher negative-review proportion (+1.52 SD), lower review activity (−0.71 SD), and somewhat lower price positioning (−0.38 SD). Its higher-volume examples generally had ratings around 2.1–2.9 stars and much greater negative-review proportions. Delivery was somewhat more common in this segment (77% versus 67%), while reservations and outdoor seating were less common.

One limitation is important: Cluster 1 also contains substantially more low-review-volume restaurants. About 50.5% of Cluster 1 restaurants had only 1–3 observed reviews, compared with 20.3% of Cluster 0. Therefore, the clusters should not be interpreted simply as "good" versus "bad" restaurants. The more defensible conclusion is that the analysis identified a higher-satisfaction/higher-engagement segment and a lower-satisfaction/lower-engagement segment, with review volume contributing to the separation.

## Check-In Activity, Prediction & Forecasting

In [ ]:
## Review check-in dataset
print(
    philly_checkins.columns.tolist()
)

print(
    philly_checkins.shape
)

philly_checkins.head()

In [ ]:
## Create event-level check-in dataset
checkin_events = philly_checkins[
    [
        'business_id',
        'date'
    ]
].copy()

## Split comma-separated check-in dates
checkin_events[
    'date'
] = (
    checkin_events[
        'date'
    ]
    .str.split(', ')
)

## Expand each check-in into its own row
checkin_events = (
    checkin_events
    .explode(
        'date'
    )
    .reset_index(
        drop=True
    )
)

checkin_events.head()

In [ ]:
## Convert check-in dates to datetime
checkin_events[
    'date'
] = pd.to_datetime(
    checkin_events[
        'date'
    ],
    format='%Y-%m-%d %H:%M:%S'
)

In [ ]:
## Review event-level check-in data
print(
    "Check-in events:",
    checkin_events.shape[0]
)

print(
    "Restaurants:",
    checkin_events[
        'business_id'
    ].nunique()
)

print(
    "First check-in:",
    checkin_events[
        'date'
    ].min()
)

print(
    "Last check-in:",
    checkin_events[
        'date'
    ].max()
)

checkin_events.head()

In [ ]:
## Calculate total check-ins by restaurant
restaurant_checkin_counts = (
    checkin_events
    .groupby(
        'business_id'
    )
    .size()
    .reset_index(
        name='total_checkins'
    )
)

restaurant_checkin_counts[
    'total_checkins'
].describe()

In [ ]:
## Review restaurants with the most check-ins
restaurant_checkin_counts = (
    restaurant_checkin_counts
    .merge(
        philly_business[
            [
                'business_id',
                'name'
            ]
        ],
        on='business_id',
        how='left'
    )
)

restaurant_checkin_counts.sort_values(
    'total_checkins',
    ascending=False
).head(10)

In [ ]:
## Create monthly check-in period
checkin_events[
    'year_month'
] = (
    checkin_events[
        'date'
    ]
    .dt.to_period('M')
)

In [ ]:
## Calculate monthly Philadelphia restaurant check-ins
monthly_checkins = (
    checkin_events
    .groupby(
        'year_month'
    )
    .size()
    .reset_index(
        name='checkins'
    )
)

## Convert monthly period to timestamp
monthly_checkins[
    'year_month'
] = (
    monthly_checkins[
        'year_month'
    ]
    .dt.to_timestamp()
)

monthly_checkins.head()

In [ ]:
## Review monthly check-in time series
print(
    "Months:",
    monthly_checkins.shape[0]
)

print(
    "First month:",
    monthly_checkins[
        'year_month'
    ].min()
)

print(
    "Last month:",
    monthly_checkins[
        'year_month'
    ].max()
)

print(
    "Total check-ins:",
    monthly_checkins[
        'checkins'
    ].sum()
)

monthly_checkins[
    'checkins'
].describe()

In [ ]:
## Visualize monthly restaurant check-ins
plt.figure(
    figsize=(14, 6)
)

plt.plot(
    monthly_checkins['year_month'],
    monthly_checkins['checkins']
)

plt.xlabel(
    'Year'
)

plt.ylabel(
    'Monthly Check-Ins'
)

plt.title(
    'Monthly Philadelphia Restaurant Check-In Activity'
)

plt.grid(
    alpha=0.3
)

plt.show()

In [ ]:
## Calculate 12-month moving average
monthly_checkins[
    'checkins_12_month_average'
] = (
    monthly_checkins[
        'checkins'
    ]
    .rolling(
        window=12
    )
    .mean()
)

In [ ]:
## Visualize monthly activity and long-term trend
plt.figure(
    figsize=(14, 6)
)

plt.plot(
    monthly_checkins[
        'year_month'
    ],
    monthly_checkins[
        'checkins'
    ],
    label='Monthly Check-Ins',
    alpha=0.5
)

plt.plot(
    monthly_checkins[
        'year_month'
    ],
    monthly_checkins[
        'checkins_12_month_average'
    ],
    label='12-Month Moving Average',
    linewidth=2
)

plt.xlabel(
    'Year'
)

plt.ylabel(
    'Check-Ins'
)

plt.title(
    'Philadelphia Restaurant Check-In Activity Over Time'
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.show()

In [ ]:
## Examine Seasonality
## Create month variable
monthly_checkins[
    'month'
] = (
    monthly_checkins[
        'year_month'
    ]
    .dt.month
)

In [ ]:
## Calculate average check-ins by month
monthly_seasonality = (
    monthly_checkins[
        monthly_checkins[
            'year_month'
        ] < '2022-01-01'
    ]
    .groupby(
        'month'
    )[
        'checkins'
    ]
    .mean()
    .reset_index()
)

monthly_seasonality

In [ ]:
## Visualize average monthly seasonality
plt.figure(
    figsize=(10, 5)
)

plt.bar(
    monthly_seasonality[
        'month'
    ],
    monthly_seasonality[
        'checkins'
    ]
)

plt.xticks(
    range(1, 13),
    [
        'Jan', 'Feb', 'Mar', 'Apr',
        'May', 'Jun', 'Jul', 'Aug',
        'Sep', 'Oct', 'Nov', 'Dec'
    ]
)

plt.xlabel(
    'Month'
)

plt.ylabel(
    'Average Check-Ins'
)

plt.title(
    'Average Philadelphia Restaurant Check-Ins by Month'
)

plt.show()

In [ ]:
## Create year variable
checkin_events[
    'year'
] = (
    checkin_events[
        'date'
    ]
    .dt.year
)

## Calculate annual check-in activity
annual_checkins = (
    checkin_events
    .groupby(
        'year'
    )
    .agg(
        total_checkins=(
            'business_id',
            'size'
        ),
        active_restaurants=(
            'business_id',
            'nunique'
        )
    )
    .reset_index()
)

annual_checkins[
    'checkins_per_active_restaurant'
] = (
    annual_checkins[
        'total_checkins'
    ]
    /
    annual_checkins[
        'active_restaurants'
    ]
)

annual_checkins

Yelp restaurant check-in activity expanded rapidly during the early years of the dataset, peaked around 2013–2014, and then entered a sustained decline well before 2020. The decline occurred both in total check-ins and check-ins per active restaurant, suggesting that it cannot be attributed solely to changes in the number of restaurants represented in the dataset. Monthly activity shows moderate seasonality, with generally higher check-in activity during spring and summer. A substantial structural break occurs in 2020, after which observed check-in activity remains far below historical levels.

In [ ]:
## Create forecasting dataset
forecasting_data = monthly_checkins[
    monthly_checkins[
        'year_month'
    ] < '2022-01-01'
].copy()

forecasting_data.shape

In [ ]:
## Create forecasting features
forecasting_data[
    'month'
] = (
    forecasting_data[
        'year_month'
    ]
    .dt.month
)

forecasting_data[
    'time_index'
] = range(
    len(forecasting_data)
)

In [ ]:
## Create lagged check-in features
forecasting_data[
    'checkins_lag_1'
] = (
    forecasting_data[
        'checkins'
    ]
    .shift(1)
)

forecasting_data[
    'checkins_lag_12'
] = (
    forecasting_data[
        'checkins'
    ]
    .shift(12)
)

In [ ]:
## Remove rows without complete lag features
forecasting_model_data = (
    forecasting_data
    .dropna(
        subset=[
            'checkins_lag_1',
            'checkins_lag_12'
        ]
    )
    .copy()
)

forecasting_model_data.head()

In [ ]:
## Create chronological training and testing datasets
forecast_training = forecasting_model_data[
    forecasting_model_data[
        'year_month'
    ] < '2019-01-01'
].copy()

forecast_testing = forecasting_model_data[
    (
        forecasting_model_data[
            'year_month'
        ] >= '2019-01-01'
    )
    &
    (
        forecasting_model_data[
            'year_month'
        ] < '2020-01-01'
    )
].copy()

print(
    "Training observations:",
    forecast_training.shape[0]
)

print(
    "Testing observations:",
    forecast_testing.shape[0]
)

print(
    "Training period:",
    forecast_training[
        'year_month'
    ].min(),
    "to",
    forecast_training[
        'year_month'
    ].max()
)

print(
    "Testing period:",
    forecast_testing[
        'year_month'
    ].min(),
    "to",
    forecast_testing[
        'year_month'
    ].max()
)

In [ ]:
## Create seasonal naive forecast
forecast_testing[
    'seasonal_naive_prediction'
] = (
    forecast_testing[
        'checkins_lag_12'
    ]
)

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

## Evaluate seasonal naive forecast
seasonal_naive_mae = mean_absolute_error(
    forecast_testing[
        'checkins'
    ],
    forecast_testing[
        'seasonal_naive_prediction'
    ]
)

seasonal_naive_rmse = np.sqrt(
    mean_squared_error(
        forecast_testing[
            'checkins'
        ],
        forecast_testing[
            'seasonal_naive_prediction'
        ]
    )
)

print(
    "Seasonal Naive MAE:",
    round(
        seasonal_naive_mae,
        2
    )
)

print(
    "Seasonal Naive RMSE:",
    round(
        seasonal_naive_rmse,
        2
    )
)

In [ ]:
## Compare actual and baseline forecast values
forecast_testing[
    [
        'year_month',
        'checkins',
        'seasonal_naive_prediction'
    ]
]

In [ ]:
from sklearn.linear_model import LinearRegression

## Create monthly dummy variables
forecasting_model_data = pd.get_dummies(
    forecasting_model_data,
    columns=[
        'month'
    ],
    prefix='month',
    drop_first=True,
    dtype=int
)

In [ ]:
## Create chronological training and testing datasets
forecast_training = forecasting_model_data[
    forecasting_model_data[
        'year_month'
    ] < '2019-01-01'
].copy()

forecast_testing = forecasting_model_data[
    (
        forecasting_model_data[
            'year_month'
        ] >= '2019-01-01'
    )
    &
    (
        forecasting_model_data[
            'year_month'
        ] < '2020-01-01'
    )
].copy()

In [ ]:
## Select forecasting predictor variables
forecast_features = [
    'checkins_lag_1',
    'checkins_lag_12',
    'time_index'
] + [
    column
    for column in forecasting_model_data.columns
    if column.startswith(
        'month_'
    )
]

forecast_features

In [ ]:
## Create training and testing variables
X_forecast_train = forecast_training[
    forecast_features
]

y_forecast_train = forecast_training[
    'checkins'
]

X_forecast_test = forecast_testing[
    forecast_features
]

y_forecast_test = forecast_testing[
    'checkins'
]

In [ ]:
## Train linear regression forecasting model
forecast_model = LinearRegression()

forecast_model.fit(
    X_forecast_train,
    y_forecast_train
)

## Predict 2019 check-in activity
forecast_testing[
    'linear_regression_prediction'
] = (
    forecast_model.predict(
        X_forecast_test
    )
)

In [ ]:
## Evaluate linear regression forecast
linear_regression_mae = mean_absolute_error(
    y_forecast_test,
    forecast_testing[
        'linear_regression_prediction'
    ]
)

linear_regression_rmse = np.sqrt(
    mean_squared_error(
        y_forecast_test,
        forecast_testing[
            'linear_regression_prediction'
        ]
    )
)

print(
    "Seasonal Naive MAE:",
    round(
        seasonal_naive_mae,
        2
    )
)

print(
    "Linear Regression MAE:",
    round(
        linear_regression_mae,
        2
    )
)

print()

print(
    "Seasonal Naive RMSE:",
    round(
        seasonal_naive_rmse,
        2
    )
)

print(
    "Linear Regression RMSE:",
    round(
        linear_regression_rmse,
        2
    )
)

In [ ]:
## Calculate improvement over baseline
mae_improvement = (
    (
        seasonal_naive_mae
        - linear_regression_mae
    )
    /
    seasonal_naive_mae
) * 100

rmse_improvement = (
    (
        seasonal_naive_rmse
        - linear_regression_rmse
    )
    /
    seasonal_naive_rmse
) * 100

print(
    "MAE Improvement:",
    round(
        mae_improvement,
        2
    ),
    "%"
)

print(
    "RMSE Improvement:",
    round(
        rmse_improvement,
        2
    ),
    "%"
)

In [ ]:
## Compare actual and predicted check-ins
forecast_comparison = forecast_testing[
    [
        'year_month',
        'checkins',
        'checkins_lag_12',
        'linear_regression_prediction'
    ]
].copy()

forecast_comparison = (
    forecast_comparison
    .rename(
        columns={
            'checkins_lag_12':
                'seasonal_naive_prediction'
        }
    )
)

forecast_comparison

In [ ]:
## Visualize 2019 forecast performance
plt.figure(
    figsize=(12, 6)
)

plt.plot(
    forecast_comparison[
        'year_month'
    ],
    forecast_comparison[
        'checkins'
    ],
    marker='o',
    label='Actual Check-Ins'
)

plt.plot(
    forecast_comparison[
        'year_month'
    ],
    forecast_comparison[
        'seasonal_naive_prediction'
    ],
    marker='o',
    label='Seasonal Naive Forecast'
)

plt.plot(
    forecast_comparison[
        'year_month'
    ],
    forecast_comparison[
        'linear_regression_prediction'
    ],
    marker='o',
    label='Linear Regression Forecast'
)

plt.xlabel(
    'Month'
)

plt.ylabel(
    'Check-Ins'
)

plt.title(
    '2019 Philadelphia Restaurant Check-In Forecast'
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.show()

Forecasting Results: A seasonal-naive forecast was established as a benchmark by using check-in activity from the corresponding month of the previous year to predict monthly activity during 2019. This approach produced an MAE of 190.25 and RMSE of 197.79 and consistently overestimated check-in activity because it did not account for the sustained decline in Yelp check-ins. A Linear Regression forecasting model was then developed using previous-month activity, activity from the same month one year earlier, long-term trend, and monthly seasonal indicators. The regression model substantially improved predictive performance, reducing MAE by 74.67% and RMSE by 70.99% relative to the seasonal-naive benchmark. The results indicate that incorporating recent activity, long-term trend, and seasonality provides substantially greater forecasting accuracy than relying on historical seasonality alone.

The model represents a rolling one-month-ahead forecast because each prediction incorporates check-in activity from the immediately preceding month. Additionally, Yelp check-ins represent platform engagement rather than total restaurant foot traffic, so the forecasts should be interpreted as predictions of Yelp check-in activity rather than overall restaurant demand.

In [ ]:
## Review current restaurant feature columns
print(
    restaurant_features.columns.tolist()
)

In [ ]:
## Review restaurant check-in activity
restaurant_features[
    'total_checkins'
].describe()

In [ ]:
## Review highest check-in activity restaurants
restaurant_features[
    [
        'name',
        'total_checkins',
        'observed_reviews',
        'average_review_rating',
        'average_sentiment',
        'negative_review_proportion',
        'price_range'
    ]
].sort_values(
    'total_checkins',
    ascending=False
).head(10)

In [ ]:
## Calculate actual check-in events by restaurant
corrected_checkin_counts = (
    checkin_events
    .groupby(
        'business_id'
    )
    .size()
    .reset_index(
        name='corrected_total_checkins'
    )
)

corrected_checkin_counts.head()

In [ ]:
## Remove incorrect check-in variable
restaurant_features = (
    restaurant_features
    .drop(
        columns='total_checkins'
    )
)

## Add corrected restaurant check-in counts
restaurant_features = (
    restaurant_features
    .merge(
        corrected_checkin_counts,
        on='business_id',
        how='left'
    )
)

In [ ]:
## Replace missing check-in counts with zero
restaurant_features[
    'corrected_total_checkins'
] = (
    restaurant_features[
        'corrected_total_checkins'
    ]
    .fillna(0)
)

In [ ]:
## Rename corrected check-in variable
restaurant_features = (
    restaurant_features
    .rename(
        columns={
            'corrected_total_checkins':
                'total_checkins'
        }
    )
)

In [ ]:
## Review corrected check-in distribution
restaurant_features[
    'total_checkins'
].describe()

In [ ]:
## Review restaurants with the most check-ins
restaurant_features[
    [
        'name',
        'total_checkins',
        'observed_reviews',
        'average_review_rating',
        'average_sentiment',
        'negative_review_proportion'
    ]
].sort_values(
    'total_checkins',
    ascending=False
).head(10)

In [ ]:
## Validate corrected restaurant check-in counts
print(
    "Restaurants with check-ins:",
    (
        restaurant_features[
            'total_checkins'
        ] > 0
    ).sum()
)

print(
    "Restaurants without check-ins:",
    (
        restaurant_features[
            'total_checkins'
        ] == 0
    ).sum()
)

print(
    "Total check-ins:",
    restaurant_features[
        'total_checkins'
    ].sum()
)

In [ ]:
## Visualize restaurant check-in distribution
plt.figure(
    figsize=(10, 5)
)

plt.hist(
    restaurant_features[
        'total_checkins'
    ],
    bins=30
)

plt.xlabel(
    'Total Check-Ins'
)

plt.ylabel(
    'Number of Restaurants'
)

plt.title(
    'Distribution of Restaurant Check-In Activity'
)

plt.show()

In [ ]:
## Create log-transformed check-in target
restaurant_features[
    'log_total_checkins'
] = np.log1p(
    restaurant_features[
        'total_checkins'
    ]
)

## Review transformed distribution
restaurant_features[
    'log_total_checkins'
].describe()

In [ ]:
## Visualize log-transformed check-in distribution
plt.figure(
    figsize=(10, 5)
)

plt.hist(
    restaurant_features[
        'log_total_checkins'
    ],
    bins=30
)

plt.xlabel(
    'Log Total Check-Ins'
)

plt.ylabel(
    'Number of Restaurants'
)

plt.title(
    'Distribution of Log-Transformed Restaurant Check-Ins'
)

plt.show()

In [ ]:
## Compare target skewness
print(
    "Raw Check-In Skewness:",
    round(
        restaurant_features[
            'total_checkins'
        ].skew(),
        2
    )
)

print(
    "Log Check-In Skewness:",
    round(
        restaurant_features[
            'log_total_checkins'
        ].skew(),
        2
    )
)

In [ ]:
## Select restaurant activity predictors
checkin_prediction_features = [
    'average_sentiment',
    'negative_review_proportion',
    'average_review_length',
    'log_observed_reviews',
    'clustering_price_range',
    'weekly_open_hours',
    'offers_delivery',
    'offers_takeout',
    'takes_reservations',
    'good_for_groups',
    'outdoor_seating'
]

In [ ]:
## Review missing prediction values
restaurant_features[
    checkin_prediction_features
].isna().sum()

In [ ]:
## Create check-in prediction dataset
checkin_prediction_data = restaurant_features[
    checkin_prediction_features
].copy()

In [ ]:
## Define numeric prediction features
numeric_checkin_features = [
    'average_sentiment',
    'negative_review_proportion',
    'average_review_length',
    'log_observed_reviews',
    'clustering_price_range',
    'weekly_open_hours'
]

## Define Boolean prediction features
boolean_checkin_features = [
    'offers_delivery',
    'offers_takeout',
    'takes_reservations',
    'good_for_groups',
    'outdoor_seating'
]

In [ ]:
## Impute missing numeric values
for column in numeric_checkin_features:

    checkin_prediction_data[
        column
    ] = (
        checkin_prediction_data[
            column
        ]
        .fillna(
            checkin_prediction_data[
                column
            ].median()
        )
    )

In [ ]:
## Impute missing Boolean values
for column in boolean_checkin_features:

    checkin_prediction_data[
        column
    ] = (
        checkin_prediction_data[
            column
        ]
        .fillna(
            checkin_prediction_data[
                column
            ].mode()[0]
        )
    )

In [ ]:
## Convert Boolean features to numeric values
for column in boolean_checkin_features:

    checkin_prediction_data[
        column
    ] = (
        checkin_prediction_data[
            column
        ]
        .astype(int)
    )

In [ ]:
## Review missing values after imputation
checkin_prediction_data.isna().sum()

In [ ]:
## Review prediction feature data types
checkin_prediction_data.dtypes

In [ ]:
## Review prepared prediction dataset
checkin_prediction_data.head()

In [ ]:
## Create check-in prediction target
checkin_prediction_target = restaurant_features[
    'log_total_checkins'
].copy()

In [ ]:
from sklearn.model_selection import train_test_split

## Create restaurant-level train and test datasets
X_checkin_train, X_checkin_test, y_checkin_train, y_checkin_test = (
    train_test_split(
        checkin_prediction_data,
        checkin_prediction_target,
        test_size=0.20,
        random_state=42
    )
)

print(
    "Training restaurants:",
    X_checkin_train.shape[0]
)

print(
    "Testing restaurants:",
    X_checkin_test.shape[0]
)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

## Train linear regression model
checkin_linear_model = LinearRegression()

checkin_linear_model.fit(
    X_checkin_train,
    y_checkin_train
)

## Predict restaurant check-in activity
checkin_linear_predictions = (
    checkin_linear_model.predict(
        X_checkin_test
    )
)

In [ ]:
## Evaluate linear regression model
checkin_linear_mae = mean_absolute_error(
    y_checkin_test,
    checkin_linear_predictions
)

checkin_linear_rmse = np.sqrt(
    mean_squared_error(
        y_checkin_test,
        checkin_linear_predictions
    )
)

checkin_linear_r2 = r2_score(
    y_checkin_test,
    checkin_linear_predictions
)

print(
    "Linear Regression MAE:",
    round(
        checkin_linear_mae,
        3
    )
)

print(
    "Linear Regression RMSE:",
    round(
        checkin_linear_rmse,
        3
    )
)

print(
    "Linear Regression R²:",
    round(
        checkin_linear_r2,
        3
    )
)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

## Train Random Forest regression model
checkin_random_forest_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

checkin_random_forest_model.fit(
    X_checkin_train,
    y_checkin_train
)

## Predict restaurant check-in activity
checkin_random_forest_predictions = (
    checkin_random_forest_model.predict(
        X_checkin_test
    )
)

In [ ]:
## Evaluate Random Forest regression model
checkin_random_forest_mae = mean_absolute_error(
    y_checkin_test,
    checkin_random_forest_predictions
)

checkin_random_forest_rmse = np.sqrt(
    mean_squared_error(
        y_checkin_test,
        checkin_random_forest_predictions
    )
)

checkin_random_forest_r2 = r2_score(
    y_checkin_test,
    checkin_random_forest_predictions
)

print(
    "Random Forest MAE:",
    round(
        checkin_random_forest_mae,
        3
    )
)

print(
    "Random Forest RMSE:",
    round(
        checkin_random_forest_rmse,
        3
    )
)

print(
    "Random Forest R²:",
    round(
        checkin_random_forest_r2,
        3
    )
)

In [ ]:
## Compare restaurant check-in prediction models
checkin_model_comparison = pd.DataFrame(
    {
        'Model': [
            'Linear Regression',
            'Random Forest'
        ],
        'MAE': [
            checkin_linear_mae,
            checkin_random_forest_mae
        ],
        'RMSE': [
            checkin_linear_rmse,
            checkin_random_forest_rmse
        ],
        'R2': [
            checkin_linear_r2,
            checkin_random_forest_r2
        ]
    }
)

checkin_model_comparison

The Random Forest explains approximately 70.7% of the variation in log-transformed restaurant check-in activity in the held-out test set, compared with 67.3% for Linear Regression. Its MAE is about 3.1% lower and RMSE about 5.3% lower. This suggests there are some nonlinear relationships or interactions in restaurant activity that Linear Regression does not fully capture, but the relatively close performance also tells us that much of the predictive signal is fairly systematic.

In [ ]:
## Review Linear Regression coefficients
checkin_linear_coefficients = pd.DataFrame(
    {
        'Feature': checkin_prediction_features,
        'Coefficient': checkin_linear_model.coef_
    }
)

checkin_linear_coefficients[
    'Absolute_Coefficient'
] = (
    checkin_linear_coefficients[
        'Coefficient'
    ].abs()
)

checkin_linear_coefficients = (
    checkin_linear_coefficients
    .sort_values(
        'Absolute_Coefficient',
        ascending=False
    )
)

checkin_linear_coefficients

In [ ]:
## Calculate Random Forest feature importance
checkin_feature_importance = pd.DataFrame(
    {
        'Feature': checkin_prediction_features,
        'Importance':
            checkin_random_forest_model.feature_importances_
    }
)

checkin_feature_importance = (
    checkin_feature_importance
    .sort_values(
        'Importance',
        ascending=False
    )
)

checkin_feature_importance

In [ ]:
## Visualize Random Forest feature importance
plt.figure(
    figsize=(10, 6)
)

plt.barh(
    checkin_feature_importance[
        'Feature'
    ],
    checkin_feature_importance[
        'Importance'
    ]
)

plt.xlabel(
    'Feature Importance'
)

plt.ylabel(
    'Restaurant Feature'
)

plt.title(
    'Random Forest Check-In Prediction Feature Importance'
)

plt.gca().invert_yaxis()

plt.show()

Restaurant-Level Check-In Prediction: Restaurant-level Yelp check-in activity was modeled using customer experience, review engagement, and restaurant characteristics. Because total check-ins were strongly right-skewed (skewness = 3.47), the target was transformed using log1p(), reducing skewness to −0.28. Linear Regression and Random Forest Regression models were evaluated using an 80/20 train-test split. Linear Regression achieved an R² of 0.673, while Random Forest improved test performance to an R² of 0.707, with lower MAE (0.807) and RMSE (1.041). These results indicate that the selected features explain a substantial portion of differences in historical Yelp check-in activity across restaurants, with modest additional predictive value from nonlinear relationships captured by Random Forest.

Feature importance showed that review volume was the dominant predictor, accounting for 70.9% of Random Forest importance. Weekly operating hours, average review length, average sentiment, and negative-review proportion provided additional predictive information. The Linear Regression model also produced a negative coefficient for negative-review proportion, suggesting that restaurants with greater concentrations of dissatisfied reviews tended to exhibit lower check-in activity when controlling for the other predictors. Overall, the results demonstrate that restaurant engagement is the strongest predictor of Yelp check-in activity, while NLP-derived measures of customer experience provide additional information. These relationships should be interpreted as predictive associations rather than causal effects.

In [ ]:
## Create prediction features without review volume
checkin_features_without_review_volume = [
    'average_sentiment',
    'negative_review_proportion',
    'average_review_length',
    'clustering_price_range',
    'weekly_open_hours',
    'offers_delivery',
    'offers_takeout',
    'takes_reservations',
    'good_for_groups',
    'outdoor_seating'
]

X_checkin_reduced = checkin_prediction_data[
    checkin_features_without_review_volume
]

## Use same train and test restaurant indices
X_checkin_reduced_train = X_checkin_reduced.loc[
    X_checkin_train.index
]

X_checkin_reduced_test = X_checkin_reduced.loc[
    X_checkin_test.index
]

In [ ]:
## Train Random Forest without review volume
checkin_random_forest_reduced = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

checkin_random_forest_reduced.fit(
    X_checkin_reduced_train,
    y_checkin_train
)

checkin_reduced_predictions = (
    checkin_random_forest_reduced.predict(
        X_checkin_reduced_test
    )
)

## Evaluate reduced Random Forest
reduced_mae = mean_absolute_error(
    y_checkin_test,
    checkin_reduced_predictions
)

reduced_rmse = np.sqrt(
    mean_squared_error(
        y_checkin_test,
        checkin_reduced_predictions
    )
)

reduced_r2 = r2_score(
    y_checkin_test,
    checkin_reduced_predictions
)

print(
    "Reduced Random Forest MAE:",
    round(reduced_mae, 3)
)

print(
    "Reduced Random Forest RMSE:",
    round(reduced_rmse, 3)
)

print(
    "Reduced Random Forest R²:",
    round(reduced_r2, 3)
)

Because review volume accounted for 70.9% of Random Forest feature importance, a reduced model was estimated without log_observed_reviews to determine whether restaurant characteristics and customer-experience measures retained predictive value independently of overall Yelp engagement. Removing review volume reduced test R² from 0.707 to 0.463, while MAE increased from 0.807 to 1.145 and RMSE increased from 1.041 to 1.411. This substantial performance decline confirms that review activity is the strongest predictor of Yelp check-in activity. However, the reduced model still explained 46.3% of test-set variation, indicating that customer-experience and restaurant characteristics retain meaningful predictive information even when review volume is excluded. These results reinforce that Yelp review and check-in activity are closely related forms of platform engagement while also demonstrating that restaurant characteristics and NLP-derived customer-experience measures contribute additional predictive signal.

## Integrate Findings and Business Recommendations


In [ ]:
## Create final restaurant analysis dataset
final_restaurant_analysis = restaurant_features[
    [
        'business_id',
        'name',
        'business_stars',
        'observed_reviews',
        'average_review_rating',
        'average_sentiment',
        'average_review_length',
        'positive_review_proportion',
        'negative_review_proportion',
        'total_checkins',
        'log_total_checkins',
        'final_cluster'
    ]
].copy()

final_restaurant_analysis.head()

In [ ]:
## Create descriptive restaurant segment names
final_restaurant_analysis[
    'restaurant_segment'
] = (
    final_restaurant_analysis[
        'final_cluster'
    ]
    .map(
        {
            0:
                'Higher-Satisfaction, Higher-Engagement',
            1:
                'Lower-Satisfaction, Lower-Engagement'
        }
    )
)

In [ ]:

## Summarize final restaurant segments
segment_summary = (
    final_restaurant_analysis
    .groupby(
        'restaurant_segment'
    )
    .agg(
        restaurants=(
            'business_id',
            'count'
        ),
        average_rating=(
            'average_review_rating',
            'mean'
        ),
        average_sentiment=(
            'average_sentiment',
            'mean'
        ),
        negative_review_proportion=(
            'negative_review_proportion',
            'mean'
        ),
        average_reviews=(
            'observed_reviews',
            'mean'
        ),
        average_checkins=(
            'total_checkins',
            'mean'
        )
    )
    .reset_index()
)

segment_summary

## Findings:

text analysis showed a clear distinction between positive and negative restaurant experiences.

Positive reviews emphasized words and phrases associated with:

* food quality
* delicious
* great
* amazing
* love
* friendly
* fresh
* great service
* highly recommend

Negative reviews shifted toward both product failures and service-process failures, including:

* bad
* bland
* dry
* overpriced
* cold
* rude
* soggy
* wait
* minute
* ask
* order
* customer service

That's an important distinction. Dissatisfaction is not simply customers saying that food is bad. The language points toward operational experiences involving ordering, waiting, staff interaction, food execution, and perceived value.

Predicting satisfaction:

The Logistic Regression classifier showed that these textual differences are systematic enough to predict customer satisfaction with strong out-of-sample performance:

Accuracy = 93.1%
ROC-AUC = 0.975

More importantly for the business problem, it correctly identified approximately 88.8% of negative reviews.

The Neural Network achieved slightly greater overall accuracy at 93.4%, but negative-review recall fell to approximately 74%. This makes the Logistic Regression model particularly useful when the objective is identifying dissatisfied customers.

Customer dissatisfaction leaves identifiable linguistic signals that could be used to systematically identify and categorize negative customer experiences.

The restaurant-level analysis extended the individual-review findings.

The final K-Means solution identified:

379 restaurants (78.3%) in the Higher-Satisfaction, Higher-Engagement segment.

105 restaurants (21.7%) in the Lower-Satisfaction, Lower-Engagement segment.

The second segment exhibited lower sentiment, greater concentrations of negative reviews, and lower review engagement.

We also discovered an important limitation: 86.7% of the lower-satisfaction/lower-engagement restaurants had 10 or fewer observed reviews, compared with approximately 44.1% in the other segment.

That means we should not interpret the clusters as simply "good restaurants" versus "bad restaurants." Instead, the segmentation identifies different customer-experience and engagement profiles, with some uncertainty for restaurants with limited review histories.

The restaurant-level Random Forest explained: R^2 = 0.707 of test-set variation in log-transformed Yelp check-in activity.

Review volume was by far the dominant predictor at 70.9% feature importance, suggesting that Yelp reviews and check-ins largely reflect a common underlying dimension of platform engagement. But the sensitivity analysis is important.

After completely removing review volume:

R^2 = 0.463

The remaining restaurant and customer-experience characteristics therefore retained substantial predictive information.

Among the full model's secondary predictors were:

* weekly operating hours
* average review length
* average sentiment
* negative-review proportion

And Linear Regression produced a negative coefficient for negative-review proportion.

What we can conclude is: Customer-experience measures derived from review data are associated with restaurant-level Yelp activity and provide predictive information beyond review volume alone.

The temporal analysis adds another dimension.

Yelp check-in activity increased substantially during the earlier portion of the dataset, peaked around 2013–2014, declined before COVID-19, and experienced a major disruption in 2020.

Importantly, check-ins per active restaurant declined as well, suggesting the downward trend cannot be explained solely by fewer restaurants appearing in the dataset.

This also explains why the seasonal-naive forecasting model performed poorly. It assumed last year's activity was a useful approximation for the current year while the underlying level of check-in activity was declining.

The rolling Linear Regression model accounted for:

* recent check-in activity
* prior-year activity
* long-term trend
* monthly seasonality

and consequently reduced MAE by 74.67% and RMSE by 70.99% relative to the seasonal-naive benchmark.

Restaurant activity contains both seasonal patterns and longer-term structural changes, making trend-aware forecasting considerably more effective than relying on historical seasonality alone.

## Business Reccomendations

1. Establish automated dissatisfaction monitoring. The Logistic Regression classifier could serve as the foundation for a review-monitoring system that flags likely negative customer experiences. Because it captured approximately 88.8% of negative reviews, restaurant managers could prioritize flagged reviews for investigation rather than manually reviewing every comment.

2. Categorize operational causes of dissatisfaction. Negative-review language repeatedly points toward ordering, waiting, staff interactions, food temperature/quality, and value. Restaurants should therefore go beyond tracking average star ratings and classify complaints into operational categories. This could allow management to distinguish food-quality problems from service-process problems and target corrective actions accordingly.

3. Track sentiment alongside conventional ratings. Star ratings provide a convenient summary, but review text contains additional information about why customers are satisfied or dissatisfied. Sentiment, negative-review proportion, and recurring phrases should therefore complement average ratings in restaurant performance dashboards.

4. Prioritize lower-satisfaction/lower-engagement restaurants for deeper diagnosis. The 105 restaurants in this segment represent a useful group for further investigation. However, because many have small review samples, businesses with adequate review histories should receive greater analytical confidence than restaurants represented by only a few reviews.

5. Treat Yelp engagement as an ecosystem rather than isolated metrics. Review volume was the strongest predictor of check-in activity, indicating that reviews and check-ins are strongly interconnected measures of Yelp engagement. Restaurants with high engagement on one dimension tend to exhibit greater activity elsewhere on the platform.

6. Use customer-experience indicators as supplementary activity signals. Even after removing review volume, the Random Forest explained 46.3% of variation in log check-in activity. This suggests that restaurant characteristics and customer-experience information can contribute to understanding engagement even when overall review volume is unavailable or deliberately excluded.

7. Incorporate trend into activity planning. Historical seasonal patterns alone substantially overpredicted 2019 check-ins. Forecasting models should therefore incorporate recent activity and longer-term trend rather than assuming previous-year platform behavior will repeat.

In [ ]:
## Visualize restaurant satisfaction and engagement
plt.figure(
    figsize=(10, 7)
)

for segment in final_restaurant_analysis[
    'restaurant_segment'
].unique():

    segment_data = final_restaurant_analysis[
        final_restaurant_analysis[
            'restaurant_segment'
        ] == segment
    ]

    plt.scatter(
        segment_data[
            'average_sentiment'
        ],
        segment_data[
            'log_total_checkins'
        ],
        label=segment,
        alpha=0.7
    )

plt.xlabel(
    'Average Review Sentiment'
)

plt.ylabel(
    'Log Total Check-Ins'
)

plt.title(
    'Restaurant Customer Sentiment and Yelp Engagement'
)

plt.legend()

plt.show()

Overall Conclusion: Analysis of 12,498 reviews across 484 Philadelphia restaurants demonstrates that unstructured customer-review data can provide actionable information about both customer experience and restaurant engagement. Positive experiences were characterized by language emphasizing food quality, service, friendliness, and recommendation, while dissatisfied customers more frequently discussed ordering, waiting, staff interactions, food-quality failures, and value. These linguistic differences were sufficiently systematic for Logistic Regression to classify positive and negative reviews with 93.1% accuracy and a ROC-AUC of 0.975 while identifying approximately 88.8% of negative reviews.

Aggregating these customer experiences to the restaurant level revealed two broader restaurant segments characterized by differences in satisfaction and engagement. Restaurant-level predictive modeling further showed that review volume was the strongest predictor of Yelp check-in activity, while sentiment, negative-review concentration, review characteristics, operating hours, and other restaurant attributes provided additional predictive information. Finally, temporal analysis demonstrated that Yelp activity reflects both seasonal behavior and substantial long-term change, with a trend-aware forecasting model dramatically outperforming a seasonal-naive benchmark. Taken together, the results demonstrate how NLP, supervised learning, unsupervised learning, and forecasting can transform unstructured customer feedback into a more comprehensive understanding of customer satisfaction, restaurant performance, and digital engagement.

In [ ]:
# HTML Rendering
!pip install jupyter
!pip install nbconvert
!jupyter nbconvert "data/Project_Danut_George.ipynb" --to html